# STEP 12. 결과 발표 슬라이드 (영어판 · 한글판)

**목적**: 고등학생도 이해할 수 있는 쉬운 말과 그림으로 결과를 설명하는 발표 자료를 만든다. 기술적 세부는 뒤쪽 **부록**에 모은다.

구성 (두 언어 동일)
- **본문 13장**
  1. 제목
  2. "가깝다 ≠ 이어졌다"
  3. **네 가지 연결 정의 한눈에 보기**(다이어그램: D 거리 · N1 꺾임 · N2 가로축 · N3 생활권)
  4. 자료
  5. 꺾임 세기
  6. 공원-길 연결
  7. N1 결과 지도
  8. N2 결과 지도
  9. N3 결과 지도
  10. 10분 규칙
  11. 거리 문제인가?
  12. 홀로 떨어진 공원
  13. 정리
- **부록 10장**
  1. 구분 슬라이드
  2. 파이프라인
  3. 보행망·연결 규칙
  4. 각도 비용
  5. N1 형식 정의
  6. N2 형식 정의
  7. N3 형식 정의
  8. 그래프 → 묶음(퍼콜레이션)
  9. 근접성 AUC
  10. 한계

구현
- 같은 코드로 `LANGS = ["en", "ko"]` 두 덱을 만든다. 글자는 `TXT` 사전에서 가져온다.
- 그림은 `figs/common`(글자 없는 아이콘·지도)과 `figs/{lang}`(라벨이 있는 그림)에 둔다.
- 차트는 PowerPoint 네이티브 차트, 흐름도·표는 네이티브 도형이라 편집할 수 있다.
- 한글 글꼴은 맑은 고딕(Malgun Gothic)을 쓴다. pptx에서는 동아시아 글꼴(`a:ea`)과 `lang="ko-KR"`까지 지정한다.
- 수치는 모두 STEP 7–11 결과 파일에서 읽는다.

출력: `outputs/{AREA}/slides/NYC_park_networks_EN.pptx`, `NYC_park_networks_KO.pptx`

In [1]:
# ===== 공통 설정 (모든 STEP 노트북에서 동일) =====
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 분석 지역: "Brooklyn"(파일럿) 또는 "NYC"(전체). 환경변수 PARK_STUDY_AREA로도 지정 가능.
STUDY_AREA = os.environ.get("PARK_STUDY_AREA", "Brooklyn")

ROOT = Path.cwd()
DATA = ROOT / "Data"
PARKS_PATH = DATA / "NYCPark" / "Parks_Properties_20260923.geojson"
NET_DIR = DATA / "derived" / "network"        # STEP1 결과 (지역 무관, NYC+버퍼 전체)
AREA_DIR = DATA / "derived" / STUDY_AREA      # STEP2 이후 결과
OUT_DIR = ROOT / "outputs" / STUDY_AREA
for _d in (NET_DIR, AREA_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

CRS = 32618            # UTM 18N (m)
R_MAX = 3200           # 탐색 한계 (m) = 보로 네트워크 버퍼 폭
SPEED = 1.0            # speed_m_s=1 -> agg_seconds == 경로 길이(m)

BOROUGHS = {"M": "Manhattan", "X": "Bronx", "B": "Brooklyn", "Q": "Queens", "R": "Staten Island"}
AREA_BOROUGHS = ["B"] if STUDY_AREA == "Brooklyn" else list(BOROUGHS)

# 네트워크 변형: C = 가로 중심선 + 보행전용 경로 (주 분석), S = 보도 포함 전체 보행망 (민감도)
VARIANTS = ["C", "S"]
PRIMARY = "C"
SIDEWALK_FOOTWAYS = {"sidewalk", "crossing", "traffic_island"}

# 공원 필터 (사용자 결정): Flagship + 비공원성 필지 제외, 면적 상한 100 acres (계산은 200 상위집합)
EXCLUDE_TYPES = ["Flagship Park", "Operations", "Lot", "Undeveloped",
                 "Buildings/Institutions", "Cemetery", "Managed Sites"]
AREA_CAP = 100
AREA_CAP_SUPERSET = 200
AREA_CAP_GRID = [50, 100, 200]
RESTRICTED_TYPES = ["Garden", "Jointly Operated Playground"]            # 접근 제한 민감도
STREET_EMBEDDED_TYPES = ["Triangle/Plaza", "Strip", "Mall", "Parkway"]  # 가로 매몰형 민감도

# 공원-네트워크 attachment
ATTACH_BUFFER = {"C": 20.0, "S": 10.0}   # 경계 버퍼 (m): C는 중심선이 도로 폭 절반만큼 떨어져 있음
ATTACH_BUFFER_GRID = {"C": [15.0, 20.0, 30.0], "S": [5.0, 10.0, 20.0]}
INSIDE_SHARE = 0.9       # 공원 내부에 90% 이상 들어간 세그먼트는 attachment에서 제외 (통행은 허용)
K_SOURCES = 8            # 공원당 출발 attachment 최대 개수 (farthest-point sampling)
SMALL_COMPONENT = 200    # 이보다 작은 연결요소에만 붙은 공원은 'unresolved'

# 최종 묶음 공통 절차: 설정별 Louvain(γ) → 보로 단위 퍼콜레이션 필터 → 합의 ≥ 0.5 → 도보 10분 지름 분할
MAIN_GAMMA = 2
COMMERCIAL_BUFFER = 100   # 연구 개요: 군집 구성 공원 주변 100 m = 상업 영역

# 묶음 크기 제약: 묶음 안 모든 공원 쌍이 서로 도보 10분 이내 (보행망 최단거리, 1.33 m/s × 600 s ≈ 800 m)
WALK_LIMIT = 800
WALK_LIMIT_GRID = [400, 800, 1200]   # 5 / 10 / 15분 민감도

# N1 angular few-turn
N1_THETAS = [15, 45, 90, 180]
N1_RADII = [800, 1600, 3200]
N1_REL_Q = [0.1, 0.2]
DIST_BANDS = [0, 200, 400, 800, 1600, 3200]

# D walking distance (근접성 기준선): 보행망 최단거리 ≤ d 이면 연결
D_DISTS = [200, 400, 800]

# N2 street-axis continuity: 같은 연속 가로(stroke) 위, 두 공원 구간 사이 간격 ≤ lim
N2_ALONG = [400, 800, 1600]
# (큰길 속성 계산용) 각도 choice/NACH 반경과 상위 비율
N2_RADII = [800, 1600, 3200]
N2_TOP_P = [0.02, 0.05, 0.10]
STROKE_ANGLE = 30
STROKE_ANGLE_GRID = [20, 30, 45]
N2_MAX_ALONG = [3200, None]

# N3 excess few-turn field overlap
N3_THETAS = [45, 90]
N3_RADII = [800, 1600]
N3_TAU = [0.1, 0.2]
N3_KAPPA = [1.0, 1.25]

print(f"STUDY_AREA={STUDY_AREA}  boroughs={AREA_BOROUGHS}  AREA_DIR={AREA_DIR.relative_to(ROOT)}")

STUDY_AREA=NYC  boroughs=['M', 'X', 'B', 'Q', 'R']  AREA_DIR=Data/derived/NYC


In [2]:
import logging
import shutil
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch, Arc
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import box, Point
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION, XL_LABEL_POSITION, XL_MARKER_STYLE
from pptx.oxml.ns import qn
from lxml import etree

logging.getLogger("cityseer").setLevel(logging.WARNING)
SL_DIR = OUT_DIR / "slides"
FIG = SL_DIR / "figs"
for sub in ["common", "en", "ko"]:
    (FIG / sub).mkdir(parents=True, exist_ok=True)
GJ = OUT_DIR / "geojson"
LANGS = ["en", "ko"]
KO_FONT = "Malgun Gothic"

# 이전 덱은 _previous로 옮김
prev = SL_DIR / "_previous"
for old in list(SL_DIR.glob("NYC_park_configurational_networks.*")):
    prev.mkdir(exist_ok=True)
    shutil.move(str(old), prev / old.name)

# ---- palette (hex without '#', for pptx) ----
DARK, FOREST, MOSS, ACCENT, ISO = "1B2E25", "2C5F2D", "97BC62", "E07A3F", "C0392B"
INK, MUTED, LIGHT = "1F2933", "5F6B73", "EEF3EA"
H = lambda c: "#" + c
GROUP_PAL = ["#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E", "#E6AB02", "#A6761D", "#1F78B4",
             "#B2DF8A", "#6A3D9A", "#33A02C", "#FF7F00", "#B15928", "#CAB2D6", "#FB9A99", "#80B1D3"]
mpl.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.unicode_minus": False,
                     "savefig.dpi": 200, "savefig.bbox": "tight", "savefig.facecolor": "white"})


def use_mpl_lang(lang):
    mpl.rcParams["font.family"] = [KO_FONT, "DejaVu Sans"] if lang == "ko" else ["Arial", "DejaVu Sans"]


use_mpl_lang("en")

## 12-1. 결과 데이터와 핵심 수치

In [3]:
pk1 = gpd.read_file(GJ / "parks_N1_angular_fewturn.geojson").to_crs(CRS).set_index("park_id", drop=False)
pk2 = gpd.read_file(GJ / "parks_N2_street_axis.geojson").to_crs(CRS).set_index("park_id", drop=False)
pkD = gpd.read_file(GJ / "parks_D_walking_distance.geojson").to_crs(CRS).set_index("park_id", drop=False)
pk3 = gpd.read_file(GJ / "parks_N3_field_overlap.geojson").to_crs(CRS).set_index("park_id", drop=False)
grp = {k: gpd.read_file(GJ / f"groups_{k}.geojson").to_crs(CRS) for k in ["N1_g2", "N2", "N3", "D"]}
CLUST_N1 = gpd.read_file(OUT_DIR / "clusters" / "clusters_N1.geojson").to_crs(CRS)
CLSUM = pd.read_csv(OUT_DIR / "clusters" / "cluster_summary.csv", index_col=0)
DCMP = pd.read_csv(OUT_DIR / "regroup" / "definition_comparison.csv")
ARI_N1N2 = float(DCMP.query("A == 'N1' and B == 'N2'").ARI.iloc[0])
ALSO_D = {r.B: r.share_of_B_pairs_also_in_A for r in DCMP[DCMP.A == "D"].itertuples()}   # N_k 묶음 쌍 중 D에서도 같은 묶음
MAIN_N2 = float(CLSUM.loc["multi-park clusters touching a top-5% main street", "N2"])
eN1 = gpd.read_file(GJ / "edges_N1.geojson").to_crs(CRS)
boroughs = gpd.read_file(NET_DIR / "boundaries.gpkg", layer="boroughs").set_index("borough").loc[AREA_BOROUGHS]
nyc_land = boroughs.union_all()
final = pd.read_csv(OUT_DIR / "regroup" / "final_group_summary.csv").set_index("method")
cross = pd.read_csv(OUT_DIR / "regroup" / "park_groups_final.csv", index_col="park_id")
verdict = pd.read_csv(AREA_DIR / "evaluation_verdict.csv", index_col=0)
ev = pd.read_csv(AREA_DIR / "evaluation_summary.csv")
WSENS = pd.read_csv(OUT_DIR / "regroup" / "walk_limit_sensitivity.csv")
parks_all = gpd.read_parquet(NET_DIR / "parks.parquet")
segC = gpd.read_parquet(NET_DIR / "segments_C.parquet", columns=["seg_id", "kind", "geometry"])

F2, FN2, FN3, FD = final.loc["N1abs γ=2"], final.loc["N2"], final.loc["N3"], final.loc["D"]
F2n, FN2n, FN3n, FDn = (final.loc["N1abs γ=2 (no walk limit)"], final.loc["N2 (no walk limit)"],
                        final.loc["N3 (no walk limit)"], final.loc["D (no walk limit)"])
raw_parks = len(gpd.read_file(PARKS_PATH, columns=["gispropnum"], ignore_geometry=True))
N = dict(raw=raw_parks, features=len(pk1), segC=len(segC),
         segS=len(pd.read_parquet(NET_DIR / "segments_S.parquet", columns=["seg_id"])),
         pieces=int((pk1.is_piece == True).sum()), split=pk1.loc[pk1.is_piece == True, "parent_id"].nunique())
N["main"] = int(parks_all.loc[~parks_all.is_piece, "in_main"].sum()) + parks_all.loc[parks_all.is_piece & parks_all.in_main, "parent_id"].nunique()
N["pairs"] = int((pd.read_parquet(AREA_DIR / f"pairs_{PRIMARY}.parquet", columns=["cfg"]).cfg
                  == f"d{int(ATTACH_BUFFER[PRIMARY])}_K{K_SOURCES}").sum())
N["iso_all"] = int((cross.cross_status == "isolated in all").sum())
N["grouped_all"] = int((cross.cross_status == "grouped in all").sum())

perc_all = pd.read_csv(AREA_DIR / "regroup" / "n1_percolation.csv", index_col="setting")
KEEP = perc_all.index[perc_all.use_for_groups].tolist()
perc = perc_all[perc_all.index.str.endswith("|best")].copy()
perc["R"] = perc.index.str.extract(r"\|R(\d+)\|", expand=False).astype(int).values
perc["theta"] = perc.index.str.extract(r"\|th(\d+)\|", expand=False).astype(int).values
PERC = perc.pivot(index="theta", columns="R", values="largest_share").sort_index()
AUC = verdict[["median_AUC", "median_AUC_all"]].reindex(["N1abs", "N1rel", "N2", "N3"])
JAC = ev.groupby("definition")["edge_jaccard_vs_null"].median()
SAME_AS_DIST = {d: 2 * JAC[d] / (1 + JAC[d]) for d in ["N1abs", "N2", "N3"]}   # 같은 개수일 때 겹치는 연결 비율 = 2J/(1+J)
ISO_B = cross[cross.cross_status == "isolated in all"].borough.value_counts().reindex(list(BOROUGHS)).fillna(0).astype(int)
display(pd.Series(N), pd.Series(ALSO_D).round(3), ISO_B, MAIN_N2)

raw               2061
features          2057
segC            495627
segS           1021976
pieces             180
split               25
main              1902
pairs           212137
iso_all            121
grouped_all         78
dtype: int64

N1    0.590
N2    0.642
N3    0.509
dtype: float64

borough
M    13
X     8
B    13
Q    52
R    35
Name: count, dtype: int64

0.465

## 12-2. 텍스트 사전 (영어 / 한국어)

In [4]:
BORO = {"en": BOROUGHS, "ko": {"M": "맨해튼", "X": "브롱크스", "B": "브루클린", "Q": "퀸스", "R": "스태튼아일랜드"}}
TXT = {
"en": {
  # figure labels
  "turn_legend": ["straight (0–15°)", "slight bend (15–45°)", "one turn (45–135°)", "two turns (135–225°)", "more turning", "parks"],
  "field_title": "Turning needed to reach each street from {park}",
  "att_legend": ["park edge zone (±20 m)", "streets along the park edge", f"{K_SOURCES} starting streets (spread out)", "park"],
  "att_title": "{name}: {n} edge streets",
  "n1_legend": ["group (colour = group)", "isolated (no partner)", "not in a stable group"],
  "zoom_legend": ["group outline", "few-turn link within 10 min", "isolated park"],
  "zoomA": "A · Harlem / Upper Manhattan – South Bronx", "zoomB": "B · Fort Greene – Bed-Stuy – Crown Heights",
  "n2_legend": ["main street (top 5% of through-walking)", "group on a shared main street", "not on a shared main street"],
  "n3_legend": ["group", "not in a group"],
  "walk_legend": ["before: one long group", "after: walkable group (colour)", "dropped: nobody within 10 min"],
  "walk_scale": "800 m ≈ 10-min walk",
  "walk_title": "{boro}: {n} parks spread over {km:.1f} km on foot → {k} walkable groups",
  "cross_labels": {"grouped in all": "grouped in all 3", "grouped in ≥2": "grouped in 2", "definition-specific": "depends on definition",
                   "isolated in ≥2": "isolated in 2", "isolated in all": "isolated in all 3"},
  "sc_titles": ["S · sidewalks & crossings drawn separately", "C · one line per street + footpaths (main)"],
  "kinds": {"sidewalk": "sidewalk", "crossing": "crossing", "traffic_island": "traffic island", "street": "street centreline",
            "service": "service road", "path": "footpath", "connector": "connector"},
  "segments": "{n:,} segments",
  "ang_titles": ["(a) Straight", "(b) One turn", "(c) Crossing the street", "(d) Cityseer dual graph"],
  "ang_c": "sidewalk network S: 180°\ncentreline network C: 0°", "parkA": "park A", "parkB": "park B", "street": "street",
  "ang_d": "segments = nodes · turns = edge costs",
  "n1c": ["A→B: 0°  (same street)", "A→C: 90°", "A→D: 180°", "E: outside R", "search bound R\n(path length)"],
  "n2c": ["one street axis\n(stroke)", "another axis", "P1–P2–P3 linked:\nsame axis, gap ≤ lim",
          "P4: different axis\n→ not linked in N2", "axis continuity\nat a junction", "< 30°: same axis", "≥ 30°: new axis"],
  "ring_legend": ["100 m commercial ring of a cluster", "ring of a single park", "parks"],
  "ring_title": "100 m commercial rings around N1 clusters (zoom B)",
  "n3c": ["few-turn fields", "metric fields", "Near pair (3 blocks)", "Far pair, same street (9 blocks)", "ratio",
          "N3 edge (τ = 0.1, κ = 1.0)", "no edge: proximity explains the overlap", "field of A", "field of B", "overlap"],
},
"ko": {
  "turn_legend": ["직진 (0–15°)", "살짝 휨 (15–45°)", "한 번 꺾음 (45–135°)", "두 번 꺾음 (135–225°)", "그 이상", "공원"],
  "field_title": "{park}에서 각 길까지 꺾어야 하는 정도",
  "att_legend": ["공원 경계 띠 (±20 m)", "공원 가장자리를 따라 난 길", f"출발점 {K_SOURCES}곳 (고르게 분산)", "공원"],
  "att_title": "{name}: 가장자리 길 {n}개",
  "n1_legend": ["묶음 (색 = 묶음)", "고립 (짝 없음)", "안정적 묶음 없음"],
  "zoom_legend": ["묶음 윤곽", "10분 이내 적게 꺾는 연결", "고립 공원"],
  "zoomA": "A · 할렘·어퍼 맨해튼 – 사우스 브롱크스", "zoomB": "B · 포트그린 – 베드스타이 – 크라운하이츠",
  "n2_legend": ["큰길 (통과 보행 상위 5%)", "같은 큰길 위 묶음", "같은 큰길 위가 아님"],
  "n3_legend": ["묶음", "묶음 없음"],
  "walk_legend": ["전: 길게 늘어진 한 묶음", "후: 걸어갈 수 있는 묶음 (색)", "제외: 10분 안에 짝 없음"],
  "walk_scale": "800 m ≈ 걸어서 10분",
  "walk_title": "{boro}: 걸어서 {km:.1f} km에 걸친 공원 {n}곳 → 걸어갈 수 있는 묶음 {k}개",
  "cross_labels": {"grouped in all": "세 정의 모두 묶임", "grouped in ≥2": "두 정의에서 묶임", "definition-specific": "정의에 따라 다름",
                   "isolated in ≥2": "두 정의에서 고립", "isolated in all": "세 정의 모두 고립"},
  "sc_titles": ["S · 보도·횡단보도를 따로 그린 망", "C · 길마다 선 하나 + 보행로 (주 분석)"],
  "kinds": {"sidewalk": "보도", "crossing": "횡단보도", "traffic_island": "교통섬", "street": "차도 중심선",
            "service": "서비스 도로", "path": "보행로", "connector": "연결선"},
  "segments": "세그먼트 {n:,}개",
  "ang_titles": ["(a) 직진", "(b) 한 번 꺾음", "(c) 길 건너기", "(d) Cityseer 듀얼 그래프"],
  "ang_c": "보도망 S: 180°\n중심선망 C: 0°", "parkA": "공원 A", "parkB": "공원 B", "street": "차도",
  "ang_d": "길 조각 = 노드 · 꺾임 = 연결 비용",
  "n1c": ["A→B: 0°  (같은 길)", "A→C: 90°", "A→D: 180°", "E: R 밖", "탐색 한계 R\n(경로 길이)"],
  "n2c": ["하나의 연속 가로\n(stroke)", "다른 가로", "P1–P2–P3 연결:\n같은 가로, 간격 ≤ lim",
          "P4: 다른 가로\n→ N2 연결 아님", "교차점에서\n연속 판정", "< 30°: 같은 가로", "≥ 30°: 새 가로"],
  "ring_legend": ["군집의 100 m 상업 영역", "단독 공원의 상업 영역", "공원"],
  "ring_title": "N1 군집 주변 100 m 상업 영역 (구역 B)",
  "n3c": ["적게 꺾는 생활권", "거리 기준 생활권", "가까운 쌍 (3블록)", "같은 길 위 먼 쌍 (9블록)", "비율",
          "N3 연결 (τ = 0.1, κ = 1.0)", "연결 아님: 가까워서 겹칠 뿐", "A의 생활권", "B의 생활권", "겹침"],
},
}

## 12-3. 그림 도우미

In [5]:
def gcolors(series):
    ids = sorted(series.dropna().unique())
    return {g: GROUP_PAL[k % len(GROUP_PAL)] for k, g in enumerate(ids)}


def legend_items(ax, items, loc="lower left", fontsize=11, **kw):
    handles = []
    for l, c, m in items:
        if m == "-":
            handles.append(Line2D([0], [0], color=c, lw=3, label=l))
        else:
            handles.append(Line2D([0], [0], marker=m, color="w", markerfacecolor=c, markeredgecolor=c, markersize=9, lw=0, label=l))
    return ax.legend(handles=handles, loc=loc, fontsize=fontsize, frameon=True, framealpha=0.92, **kw)


def overview(ax, pk, gcol, zooms=(), dark=False, corridors=None, show_iso=True):
    boroughs.plot(ax=ax, color="#24392E" if dark else "#F1F3EF", edgecolor="#3D5446" if dark else "white", linewidth=0.8)
    if corridors is not None:
        corridors.plot(ax=ax, color="#8E3B2F", linewidth=0.35, alpha=0.7)
    pts = pk.geometry.representative_point()
    size = np.clip(np.sqrt(pk.piece_acres.astype(float)) * 7, 3, 45)
    grouped = pk[gcol].notna()
    iso = (pk.status == "robust_isolated") & show_iso
    rest = ~grouped & ~iso
    ax.scatter(pts[rest].x, pts[rest].y, s=4, c="#8C9A92" if dark else "#BDBDBD", lw=0)
    cmap = gcolors(pk[gcol])
    ax.scatter(pts[grouped].x, pts[grouped].y, s=size[grouped], c=pk.loc[grouped, gcol].map(cmap), lw=0, alpha=0.95)
    ax.scatter(pts[iso].x, pts[iso].y, s=14, c=H(ISO), marker="x", lw=1.0)
    for label, bb in zooms:
        ax.add_patch(Rectangle((bb[0], bb[1]), bb[2] - bb[0], bb[3] - bb[1], fill=False, ec=H(INK), lw=1.4))
        ax.text(bb[0], bb[3] + 300, label, fontsize=11, fontweight="bold", color=H(INK))
    ax.set_aspect("equal")
    ax.set_axis_off()


def zoom(ax, pk, gcol, groups=None, bb=None, edges=None, corridors=None, show_iso=True):
    win = box(*bb)
    segC.iloc[segC.sindex.query(win, predicate="intersects")].plot(ax=ax, color="#DADDD8", linewidth=0.5)
    if corridors is not None:
        corridors[corridors.intersects(win)].plot(ax=ax, color="#8E3B2F", linewidth=1.8, alpha=0.6)
    sub = pk[pk.intersects(win)]
    cmap = gcolors(pk[gcol])
    iso_m = (sub.status == "robust_isolated") & show_iso
    sub[sub[gcol].isna() & ~iso_m].plot(ax=ax, color="#E3E6E1", edgecolor="#AEB5AF", lw=0.4)
    g = sub[sub[gcol].notna()]
    if len(g):
        g.plot(ax=ax, color=g[gcol].map(cmap), edgecolor="white", lw=0.3)
    if iso_m.any():
        sub[iso_m].plot(ax=ax, color="#F6D5D1", edgecolor=H(ISO), lw=1.2)
    if edges is not None:
        edges[edges.intersects(win) & (edges.share >= 0.5) & edges.within_walk_limit].plot(ax=ax, color=H(ACCENT), lw=0.6, alpha=0.6)
    if groups is not None:
        for _, r in groups[groups.intersects(win)].iterrows():
            gpd.GeoSeries([r.geometry.boundary], crs=CRS).plot(ax=ax, color=cmap.get(r.group_id, "#777777"), lw=0.9, ls="--")
    ax.set_xlim(bb[0], bb[2]); ax.set_ylim(bb[1], bb[3])
    ax.set_aspect("equal"); ax.set_axis_off()


def to_utm_bbox(lon0, lat0, lon1, lat1):
    return tuple(gpd.GeoSeries([box(lon0, lat0, lon1, lat1)], crs=4326).to_crs(CRS).total_bounds)


ZOOM_A = to_utm_bbox(-73.962, 40.796, -73.898, 40.842)
ZOOM_B = to_utm_bbox(-73.968, 40.664, -73.904, 40.710)


def street_grid(ax, n=7, lw=1.0, color="#C9CEC8"):
    for k in range(n + 1):
        ax.plot([0, n], [k, k], color=color, lw=lw, zorder=1)
        ax.plot([k, k], [0, n], color=color, lw=lw, zorder=1)


def park(ax, x, y, label, color=H(FOREST), w=0.6, fs=12):
    ax.add_patch(Rectangle((x + (1 - w) / 2, y + (1 - w) / 2), w, w, color=color, zorder=3))
    ax.text(x + 0.5, y + 0.5, label, ha="center", va="center", color="white", fontsize=fs, fontweight="bold", zorder=4)


def route(ax, pts, color=H(ACCENT), lw=4, ls="-"):
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=lw, zorder=2, solid_capstyle="round", ls=ls)


def dot(ax, x, y, color=H(ACCENT), r=0.1):
    ax.add_patch(Circle((x, y), r, color=color, zorder=5))


def mark(ax, x, y, ok, fs=26):
    ax.text(x, y, "✓" if ok else "✗", fontsize=fs, fontweight="bold", color=H(FOREST) if ok else H(ISO),
            ha="center", va="center", family="DejaVu Sans", zorder=6,
            path_effects=[pe.withStroke(linewidth=4, foreground="white")])


def finish(ax, xlim, ylim):
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect("equal"); ax.set_axis_off()

## 12-4. 언어 공통 그림 (글자 없는 아이콘·지도)
- 정의 아이콘 4개(D 가까운 거리, N1 적게 꺾는 길, N2 같은 길(축), N3 겹치는 생활권)
- "가깝다 ≠ 이어졌다" 대비 그림, 꺾임 아이콘, 보행망 예시, 제목 지도

In [6]:
C = FIG / "common"
# --- N1 icon: straight link vs two-turn route
fig, ax = plt.subplots(figsize=(4.2, 4.2))
street_grid(ax, 5)
route(ax, [(1, 2), (4, 2)])
route(ax, [(1, 3), (2, 3), (2, 4), (3, 4)], color="#9AA39C", lw=3, ls=(0, (2, 1.5)))
dot(ax, 2, 3, "#9AA39C"); dot(ax, 2, 4, "#9AA39C")
park(ax, 0, 2, "A", color=H(DARK)); park(ax, 4, 2, "B"); park(ax, 3, 4, "C", color="#9AA39C")
ax.text(2.5, 1.62, "0°", fontsize=17, fontweight="bold", color=H(ACCENT), ha="center")
ax.text(1.35, 4.25, "180°", fontsize=14, fontweight="bold", color="#7E877F", ha="center")
mark(ax, 4.5, 1.25, True); mark(ax, 3.5, 5.6, False)
finish(ax, (-0.2, 5.2), (0.3, 6.1))
fig.savefig(C / "def_N1.png"); plt.close(fig)

# --- N2 icon: parks on the same continuous street axis
fig, ax = plt.subplots(figsize=(4.2, 4.2))
street_grid(ax, 5)
axis_ = [(0, 0.6), (2.5, 2.0), (5, 3.4)]
ax.plot(*zip(*axis_), color=H(FOREST), lw=7, alpha=0.9, zorder=2, solid_capstyle="round")
ax.plot([3, 3], [0, 5], color="#8C968E", lw=4, zorder=2)
for x, y, lab in [(0.55, 1.05, "1"), (2.2, 2.35, "2"), (4.0, 3.15, "3")]:
    ax.add_patch(Rectangle((x, y), 0.55, 0.5, color=H(MOSS), ec=H(FOREST), lw=1.5, zorder=4))
    ax.text(x + 0.27, y + 0.25, lab, color=H(DARK), ha="center", va="center", fontsize=12, fontweight="bold", zorder=5)
for a, b in [((0.85, 1.6), (2.45, 2.9)), ((2.45, 2.9), (4.3, 3.7))]:
    ax.add_patch(FancyArrowPatch(a, b, connectionstyle="arc3,rad=-0.4", arrowstyle="-", color=H(ACCENT), lw=2.5, ls="--", zorder=6))
ax.add_patch(Rectangle((3.1, 0.6), 0.55, 0.5, color="#9AA39C", zorder=4))
ax.text(3.37, 0.85, "4", color="white", ha="center", va="center", fontsize=12, fontweight="bold", zorder=5)
mark(ax, 1.4, 4.4, True); mark(ax, 4.3, 0.85, False)
finish(ax, (-0.2, 5.2), (-0.2, 5.4))
fig.savefig(C / "def_N2.png"); plt.close(fig)

# --- D icon: within a short walk
fig, ax = plt.subplots(figsize=(4.2, 4.2))
street_grid(ax, 5)
ax.add_patch(Circle((1.5, 2.5), 2.1, fill=False, ls=(0, (4, 3)), ec=H(MUTED), lw=1.6))
route(ax, [(1, 2), (2, 2), (2, 1)], lw=4)
park(ax, 1, 2, "A", color=H(DARK)); park(ax, 2, 0, "B"); park(ax, 4, 4, "C", color="#9AA39C")
ax.text(2.35, 1.55, "400 m", fontsize=13, fontweight="bold", color=H(ACCENT),
        path_effects=[pe.withStroke(linewidth=3, foreground="white")])
mark(ax, 2.5, -0.35, True); mark(ax, 4.5, 3.55, False)
finish(ax, (-0.2, 5.2), (-0.8, 5.2))
fig.savefig(C / "def_D.png"); plt.close(fig)

# --- N3 icon: overlapping straight-walk zones
fig, ax = plt.subplots(figsize=(4.2, 4.2))
NGI, RS = 7, 3
segs_i = [((x, y), (x + 1, y)) for y in range(NGI + 1) for x in range(NGI)] + \
         [((x, y), (x, y + 1)) for x in range(NGI + 1) for y in range(NGI)]


def zone(px, py):
    f = set()
    for s in segs_i:
        (x0, y0), (x1, y1) = s
        if y0 == y1 and y0 in (py, py + 1) and max(x0, x1) <= px + 1 + RS and min(x0, x1) >= px - RS:
            f.add(s)
        if x0 == x1 and x0 in (px, px + 1) and max(y0, y1) <= py + 1 + RS and min(y0, y1) >= py - RS:
            f.add(s)
    return f


za, zb = zone(1, 3), zone(4, 3)
for s in segs_i:
    c, lw = "#E4E7E2", 1.0
    if s in za and s in zb:
        c, lw = "#6A3D9A", 5
    elif s in za:
        c, lw = H(FOREST), 3
    elif s in zb:
        c, lw = H(ACCENT), 3
    ax.plot([s[0][0], s[1][0]], [s[0][1], s[1][1]], color=c, lw=lw, solid_capstyle="butt")
park(ax, 1, 3, "A", color=H(DARK)); park(ax, 4, 3, "B", color=H(DARK))
mark(ax, 3.0, 5.0, True)
finish(ax, (-0.2, 7.2), (-0.2, 7.2))
fig.savefig(C / "def_N3.png"); plt.close(fig)

# --- close vs connected
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
ax = axes[0]
street_grid(ax, 4)
ax.add_patch(Rectangle((-0.1, 1.85), 4.2, 0.3, color="#6F7A72", zorder=2))
route(ax, [(1, 1), (4, 1), (4, 3), (1, 3)], lw=3.5, ls=(0, (2, 1.2)))
for x, y in [(4, 1), (4, 3)]:
    dot(ax, x, y)
park(ax, 0, 1, "A", color=H(DARK)); park(ax, 0, 2.2, "B", w=0.55)
ax.plot([0.5, 0.5], [1.75, 2.42], color=H(INK), lw=1.2, ls=":")
ax.text(0.62, 2.06, "150 m", fontsize=11, color=H(INK), va="center", fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")])
mark(ax, 2.5, 3.6, False)
finish(ax, (-0.2, 4.3), (-0.2, 4.2))
ax = axes[1]
for y in [0, 1, 2]:
    ax.plot([0, 9], [y, y], color="#C9CEC8", lw=1)
for x in range(10):
    ax.plot([x, x], [0, 2], color="#C9CEC8", lw=1)
route(ax, [(1, 1), (8, 1)])
park(ax, 0, 1, "A", color=H(DARK)); park(ax, 8, 1, "C")
ax.annotate("", xy=(8, 2.35), xytext=(1, 2.35), arrowprops=dict(arrowstyle="<->", color=H(INK), lw=1.2))
ax.text(4.5, 2.5, "1 km", fontsize=12, color=H(INK), ha="center", fontweight="bold")
mark(ax, 4.5, 0.45, True)
finish(ax, (-0.2, 9.2), (-0.4, 3.0))
plt.tight_layout()
fig.savefig(C / "close_vs_connected.png"); plt.close(fig)

# --- turn icons
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
for ax, pts, lab in zip(axes, [[(0, 0), (3, 0)], [(0, 0), (2, 0), (2, 2)], [(0, 0), (1.5, 0), (1.5, 1.5), (3, 1.5)]],
                        ["0°", "90°", "180°"]):
    route(ax, pts, color=H(FOREST), lw=6)
    for p in pts[1:-1]:
        dot(ax, *p, color=H(ACCENT), r=0.16)
    ax.text(1.5, -0.75, lab, fontsize=22, fontweight="bold", color=H(ACCENT), ha="center")
    finish(ax, (-0.4, 3.4), (-1.2, 2.4))
plt.tight_layout()
fig.savefig(C / "turn_icons.png"); plt.close(fig)

# --- walking network + parks (Fort Greene)
center = gpd.GeoSeries([Point(-73.9754, 40.6911)], crs=4326).to_crs(CRS).iloc[0]
win = center.buffer(700).envelope
fig, ax = plt.subplots(figsize=(6, 6))
segC.iloc[segC.sindex.query(win, predicate="intersects")].clip(win).plot(ax=ax, color="#5F6B73", lw=0.7)
pv = parks_all[parks_all.intersects(win) & ~parks_all.is_piece].clip(win)
pv.plot(ax=ax, color="#9CC77A", edgecolor=H(FOREST), lw=0.6, alpha=0.9)
ax.set_xlim(win.bounds[0], win.bounds[2]); ax.set_ylim(win.bounds[1], win.bounds[3]); ax.set_axis_off()
fig.savefig(C / "network_parks.png"); plt.close(fig)

# --- dark title map
fig, ax = plt.subplots(figsize=(7, 8), facecolor=H(DARK))
overview(ax, pk1, "group_g2", dark=True)
fig.savefig(C / "title_map.png", facecolor=H(DARK)); plt.close(fig)
print("common figures:", sorted(p.name for p in C.iterdir()))

common figures: ['close_vs_connected.png', 'def_D.png', 'def_N1.png', 'def_N2.png', 'def_N3.png', 'network_parks.png', 'title_map.png', 'turn_icons.png']


## 12-5. 데이터 준비: 각도 장(angular field), 10분 분할 예시, 회랑

In [7]:
from cityseer.network import CityNetwork
att = pd.read_parquet(NET_DIR / "attachments.parquet")
focus = parks_all[parks_all.name.str.contains("Maria Hernandez", na=False)].iloc[0]
fa = att[(att.variant == PRIMARY) & att.is_default & (att.park_id == focus.park_id)]
cn = CityNetwork.load(NET_DIR / "cityseer" / f"cn_{PRIMARY}_{focus.borough}")
ns, ng = cn.network_structure, cn.nodes_gdf
best = np.full(int(ng.ns_node_idx.max()) + 1, np.inf)
for sid in fa.loc[fa.fps_rank < K_SOURCES, "seg_id"]:
    if sid in ng.index:
        vis, tree = ns.dijkstra_tree_simplest(int(ng.loc[sid, "ns_node_idx"]), 1600, SPEED)
        for t in vis:
            best[t] = min(best[t], tree[t].simpl_dist)
FIELD = cn.to_geopandas()
FIELD["ang"] = best[FIELD["ns_node_idx"].values]
FIELD = FIELD[np.isfinite(FIELD.ang)][["ang", "geometry"]]
del cn, ns

# 10분 분할 예시
pw = pd.read_parquet(AREA_DIR / f"pairs_{PRIMARY}.parquet", columns=["cfg", "i", "j", "D_net"])
pw = pw[pw.cfg == f"d{int(ATTACH_BUFFER[PRIMARY])}_K{K_SOURCES}"]
DNET = pw.assign(a=np.minimum(pw.i, pw.j), b=np.maximum(pw.i, pw.j)).groupby(["a", "b"]).D_net.min().to_dict()
del pw
wd = lambda x, y: 0.0 if x == y else DNET.get((x, y) if x < y else (y, x), np.inf)
cand = []
for gid, m in pk1.groupby("group_g2_nolimit").groups.items():
    m = list(m)
    if 8 <= len(m) <= 25:
        cand.append((gid, len(m), max(wd(a, b) for k, a in enumerate(m) for b in m[k + 1:]), pk1.loc[m, "group_g2"].dropna().nunique()))
cand = pd.DataFrame(cand, columns=["gid", "n", "max_walk", "sub"])
EX = cand[np.isfinite(cand.max_walk)].sort_values(["sub", "max_walk"], ascending=False).iloc[0]
EXM = pk1[pk1.group_g2_nolimit == EX.gid]
EXN = dict(n=int(EX.n), km=EX.max_walk / 1000, k=int(EX["sub"]), dropped=int(EXM.group_g2.isna().sum()),
           boro=EXM.borough.mode().iloc[0])

# 회랑 (top 5% NACH-1600 strokes)
strokes = pd.read_parquet(AREA_DIR / f"strokes_{PRIMARY}.parquet", columns=["seg_id", f"stroke_{STROKE_ANGLE}"])
corr = pd.read_parquet(AREA_DIR / f"corridors_{PRIMARY}.parquet")
sel = corr[(corr.angle == STROKE_ANGLE) & (corr.r == 1600) & (corr.p == 0.05)].stroke
CORR = segC[segC.seg_id.isin(strokes.loc[strokes[f"stroke_{STROKE_ANGLE}"].isin(sel), "seg_id"])]
CORR = CORR[CORR.intersects(nyc_land)]
EXN

Parsing geometries:   0%|          | 0/166522 [00:00<?, ?it/s]

Parsing geometries:   4%|▍         | 7295/166522 [00:00<00:02, 72939.11it/s]

Parsing geometries:   9%|▉         | 14725/166522 [00:00<00:02, 73738.70it/s]

Parsing geometries:  13%|█▎        | 22099/166522 [00:00<00:01, 73483.34it/s]

Parsing geometries:  18%|█▊        | 29645/166522 [00:00<00:01, 74259.14it/s]

...

Building edges:  95%|█████████▌| 267086/280055 [00:04<00:00, 62195.90it/s]

Building edges:  98%|█████████▊| 273311/280055 [00:04<00:00, 61616.47it/s]

Building edges: 100%|█████████▉| 279477/280055 [00:04<00:00, 60448.06it/s]

Building edges: 100%|██████████| 280055/280055 [00:04<00:00, 61436.64it/s]

{'n': 17,
 'km': np.float64(2.21855615234375),
 'k': 5,
 'dropped': 1,
 'boro': 'B'}

## 12-6. 언어별 그림 (라벨 포함)

In [8]:
def make_lang_figs(lang):
    use_mpl_lang(lang)
    T, D = TXT[lang], FIG / lang
    # angular field
    bins = [0, 15, 45, 135, 225, 10000]
    cols = ["#08306B", "#2171B5", "#6BAED6", "#F2A65A", "#E3E3E3"]
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    FIELD.sort_values("ang", ascending=False).plot(ax=ax, column="ang", cmap=ListedColormap(cols), norm=BoundaryNorm(bins, 5), lw=0.9)
    parks_all[parks_all.intersects(box(*FIELD.total_bounds)) & ~parks_all.is_piece].plot(ax=ax, color="#CFE3B8", edgecolor=H(FOREST), lw=0.5)
    gpd.GeoSeries([focus.geometry], crs=CRS).plot(ax=ax, color=H(ISO))
    legend_items(ax, list(zip(T["turn_legend"], cols[:4] + ["#C8C8C8", "#CFE3B8"], ["-"] * 5 + ["s"])), loc="lower left", fontsize=10.5)
    ax.set_title(T["field_title"].format(park=focus["name"]), fontsize=12.5); ax.set_axis_off()
    fig.savefig(D / "angular_field.png"); plt.close(fig)
    # attachment
    win = focus.geometry.buffer(110).envelope
    segs = segC.iloc[segC.sindex.query(win, predicate="intersects")]
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    gpd.GeoSeries([focus.geometry.boundary.buffer(ATTACH_BUFFER[PRIMARY])], crs=CRS).plot(ax=ax, color="#FBE3D3")
    gpd.GeoSeries([focus.geometry], crs=CRS).plot(ax=ax, color="#CFE3B8", edgecolor=H(FOREST), lw=1.2)
    segs.plot(ax=ax, color="#BFC5BF", lw=0.8)
    segs[segs.seg_id.isin(fa.seg_id)].plot(ax=ax, color=H(ACCENT), lw=1.8)
    segs[segs.seg_id.isin(fa.loc[fa.fps_rank < K_SOURCES, "seg_id"])].plot(ax=ax, color=H(DARK), lw=3.8)
    ax.set_xlim(win.bounds[0], win.bounds[2]); ax.set_ylim(win.bounds[1], win.bounds[3])
    legend_items(ax, list(zip(T["att_legend"], ["#FBE3D3", H(ACCENT), H(DARK), "#CFE3B8"], ["s", "-", "-", "s"])), loc="lower right", fontsize=10)
    ax.set_title(T["att_title"].format(name=focus["name"], n=len(fa)), fontsize=12); ax.set_axis_off()
    fig.savefig(D / "attachment.png"); plt.close(fig)
    # N1 maps
    fig, ax = plt.subplots(figsize=(7.2, 8))
    overview(ax, pk1, "group_g2", zooms=[("A", ZOOM_A)])
    legend_items(ax, list(zip(T["n1_legend"], [GROUP_PAL[0], H(ISO), "#BDBDBD"], ["o", "X", "o"])), loc="upper left", fontsize=11)
    fig.savefig(D / "N1_overview.png"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 7))
    zoom(ax, pk1, "group_g2", grp["N1_g2"], ZOOM_A, edges=eN1)
    legend_items(ax, list(zip(T["zoom_legend"], ["#555555", H(ACCENT), H(ISO)], ["-", "-", "s"])), loc="lower right", fontsize=10.5)
    ax.set_title(T["zoomA"], fontsize=12.5)
    fig.savefig(D / "N1_zoom.png"); plt.close(fig)
    # D maps
    fig, ax = plt.subplots(figsize=(7.2, 8))
    overview(ax, pkD, "group_id", zooms=[("B", ZOOM_B)])
    legend_items(ax, list(zip(T["n1_legend"], [GROUP_PAL[0], H(ISO), "#BDBDBD"], ["o", "X", "o"])), loc="upper left", fontsize=11)
    fig.savefig(D / "D_overview.png"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 7))
    zoom(ax, pkD, "group_id", grp["D"], ZOOM_B)
    ax.set_title(T["zoomB"], fontsize=12.5)
    fig.savefig(D / "D_zoom.png"); plt.close(fig)
    # N2 maps (street-axis continuity)
    fig, ax = plt.subplots(figsize=(7.2, 8))
    overview(ax, pk2, "group_id", zooms=[("B", ZOOM_B)])
    legend_items(ax, list(zip(T["n1_legend"], [GROUP_PAL[0], H(ISO), "#BDBDBD"], ["o", "X", "o"])), loc="upper left", fontsize=11)
    fig.savefig(D / "N2_overview.png"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 7))
    zoom(ax, pk2, "group_id", grp["N2"], ZOOM_B)
    ax.set_title(T["zoomB"], fontsize=12.5)
    fig.savefig(D / "N2_zoom.png"); plt.close(fig)
    # N3 maps
    fig, ax = plt.subplots(figsize=(7.2, 8))
    overview(ax, pk3, "group_id", zooms=[("B", ZOOM_B)], show_iso=False)
    legend_items(ax, list(zip(T["n3_legend"], [GROUP_PAL[0], "#BDBDBD"], ["o", "o"])), loc="upper left", fontsize=11)
    fig.savefig(D / "N3_overview.png"); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 7))
    zoom(ax, pk3, "group_id", grp["N3"], ZOOM_B, show_iso=False)
    ax.set_title(T["zoomB"], fontsize=12.5)
    fig.savefig(D / "N3_zoom.png"); plt.close(fig)
    # commercial rings preview (N1 clusters, zoom B)
    win = box(*ZOOM_B)
    sub = CLUST_N1[CLUST_N1.intersects(win)]
    multi = sub[~sub.is_singleton]
    fig, ax = plt.subplots(figsize=(7, 7))
    segC.iloc[segC.sindex.query(win, predicate="intersects")].plot(ax=ax, color="#DADDD8", lw=0.5)
    sub[sub.is_singleton].plot(ax=ax, color="#E3E3E3", edgecolor="#C4C4C4", lw=0.3)
    multi.plot(ax=ax, color=[GROUP_PAL[k % len(GROUP_PAL)] for k in range(len(multi))], alpha=0.7, edgecolor="white", lw=0.3)
    pk1[pk1.intersects(win)].plot(ax=ax, color=H(FOREST), alpha=0.85)
    legend_items(ax, list(zip(T["ring_legend"], [GROUP_PAL[0], "#E3E3E3", H(FOREST)], ["s", "s", "s"])), loc="lower right", fontsize=10.5)
    ax.set_xlim(win.bounds[0], win.bounds[2]); ax.set_ylim(win.bounds[1], win.bounds[3]); ax.set_axis_off()
    ax.set_title(T["ring_title"], fontsize=12.5)
    fig.savefig(D / "rings.png"); plt.close(fig)
    # walk split example
    hull = EXM.union_all().convex_hull
    bb = hull.buffer(250).bounds
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    segC.iloc[segC.sindex.query(box(*bb), predicate="intersects")].plot(ax=ax, color="#DADDD8", lw=0.5)
    gpd.GeoSeries([hull.boundary], crs=CRS).plot(ax=ax, color=H(INK), lw=1.2, ls="--")
    cm = gcolors(EXM.group_g2)
    EXM[EXM.group_g2.isna()].plot(ax=ax, color="#C9CDC8", edgecolor="#8E978F", lw=0.6)
    gg = EXM[EXM.group_g2.notna()]
    gg.plot(ax=ax, color=gg.group_g2.map(cm), edgecolor="white", lw=0.4)
    for gid, g in gg.groupby("group_g2"):
        gpd.GeoSeries([g.union_all().convex_hull.buffer(40).boundary], crs=CRS).plot(ax=ax, color=cm[gid], lw=1.6)
    x0, y0 = bb[0] + 150, bb[1] + 120
    ax.plot([x0, x0 + WALK_LIMIT], [y0, y0], color=H(INK), lw=3)
    ax.text(x0 + WALK_LIMIT / 2, y0 + 60, T["walk_scale"], ha="center", fontsize=11, color=H(INK))
    lg = legend_items(ax, list(zip(T["walk_legend"], [H(INK), GROUP_PAL[0], "#C9CDC8"], ["-", "s", "s"])),
                      loc="upper center", fontsize=10.5, bbox_to_anchor=(0.5, -0.01))
    ax.set_xlim(bb[0], bb[2]); ax.set_ylim(bb[1], bb[3]); ax.set_aspect("equal"); ax.set_axis_off()
    ax.set_title(T["walk_title"].format(boro=BORO[lang][EXN["boro"]], n=EXN["n"], km=EXN["km"], k=EXN["k"]), fontsize=12)
    fig.savefig(D / "walk_split.png"); plt.close(fig)
    # cross status
    cc = {"grouped in all": "#1A9850", "grouped in ≥2": "#91CF60", "definition-specific": "#BDBDBD",
          "isolated in ≥2": "#FC8D59", "isolated in all": H(ISO)}
    fig, ax = plt.subplots(figsize=(7.2, 8))
    boroughs.plot(ax=ax, color="#F1F3EF", edgecolor="white", lw=0.8)
    pts = pk1.geometry.representative_point()
    stt = cross.reindex(pk1.index).cross_status
    for k, c in cc.items():
        m = (stt == k).values
        ax.scatter(pts[m].x, pts[m].y, s=14 if "all" in k else 6, c=c, lw=0, label=f"{T['cross_labels'][k]} ({int(m.sum())})")
    ax.legend(loc="upper left", fontsize=11, framealpha=0.92, markerscale=1.8)
    ax.set_aspect("equal"); ax.set_axis_off()
    fig.savefig(D / "cross_status.png"); plt.close(fig)
    # appendix: S vs C
    segS = gpd.read_parquet(NET_DIR / "segments_S.parquet", columns=["kind", "geometry"])
    win = gpd.GeoSeries([Point(-73.9754, 40.6911)], crs=4326).to_crs(CRS).iloc[0].buffer(550).envelope
    KC = {"sidewalk": "#1F78B4", "crossing": H(ISO), "traffic_island": "#6A3D9A", "street": "#333333",
          "service": "#8C6D5A", "path": H(FOREST), "connector": H(ACCENT)}
    fig, axes = plt.subplots(1, 2, figsize=(12, 6.4))
    for ax, ttl, gdf in zip(axes, T["sc_titles"], [segS, segC]):
        sub = gdf.iloc[gdf.sindex.query(win, predicate="intersects")].clip(win)
        for k, g2 in sub.groupby("kind"):
            g2.plot(ax=ax, color=KC.get(k, "grey"), linewidth=1.2 if k == "street" else 0.9, label=T["kinds"].get(k, k))
        ax.set_title(f"{ttl}\n{T['segments'].format(n=len(sub))}", fontsize=14)
        ax.legend(loc="lower left", fontsize=11.5, framealpha=0.92)
        ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(D / "network_S_vs_C.png"); plt.close(fig)
    del segS
    # appendix: angular concept
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
    ax = axes[0]
    for x in range(4):
        ax.plot([x, x + 1], [0, 0], color="#9AA39C", lw=6, solid_capstyle="butt")
    ax.plot([0, 1], [0, 0], color=H(FOREST), lw=6); ax.plot([3, 4], [0, 0], color=H(ACCENT), lw=6)
    ax.annotate("", xy=(3.5, 0.25), xytext=(0.5, 0.25), arrowprops=dict(arrowstyle="->", color=H(INK), lw=1.5))
    ax.text(2, 0.55, "Σ = 0°", ha="center", fontsize=13, fontweight="bold", color=H(INK))
    ax.set_xlim(-0.3, 4.3); ax.set_ylim(-1.2, 1.4)
    ax = axes[1]
    ax.plot([0, 2], [0, 0], color="#9AA39C", lw=6); ax.plot([2, 2], [0, 2], color="#9AA39C", lw=6)
    ax.plot([0, 1], [0, 0], color=H(FOREST), lw=6); ax.plot([2, 2], [1, 2], color=H(ACCENT), lw=6)
    ax.add_patch(Arc((2, 0), 0.6, 0.6, theta1=90, theta2=180, color=H(ACCENT), lw=2))
    ax.text(1.55, 0.35, "90°", fontsize=12, color=H(ACCENT), fontweight="bold")
    ax.text(1.0, 1.6, "Σ = 90°", ha="center", fontsize=13, fontweight="bold", color=H(INK))
    ax.set_xlim(-0.3, 2.6); ax.set_ylim(-0.6, 2.3)
    ax = axes[2]
    ax.add_patch(Rectangle((0, 0.35), 3.2, 0.5, color="#ECEDEA"))
    ax.text(3.15, 0.6, T["street"], fontsize=9, color=H(MUTED), ha="right", va="center")
    ax.plot([0, 3.2], [0.3, 0.3], color="#9AA39C", lw=4); ax.plot([0, 3.2], [0.9, 0.9], color="#9AA39C", lw=4)
    ax.plot([1.6, 1.6], [0.3, 0.9], color="#9AA39C", lw=4, ls=(0, (1, 1)))
    ax.plot([0.2, 1.6], [0.3, 0.3], color=H(FOREST), lw=4); ax.plot([0.2, 1.6], [0.9, 0.9], color=H(ACCENT), lw=4)
    ax.add_patch(Rectangle((0.3, -0.45), 1.0, 0.6, color=H(MOSS))); ax.text(0.8, -0.15, T["parkA"], ha="center", va="center", fontsize=9)
    ax.add_patch(Rectangle((0.3, 1.05), 1.0, 0.6, color=H(MOSS))); ax.text(0.8, 1.35, T["parkB"], ha="center", va="center", fontsize=9)
    for yy in (0.3, 0.9):
        dot(ax, 1.6, yy, r=0.07)
    ax.text(1.72, 0.12, "90°", fontsize=11, color=H(ACCENT), fontweight="bold"); ax.text(1.72, 0.98, "90°", fontsize=11, color=H(ACCENT), fontweight="bold")
    ax.text(1.6, 2.05, T["ang_c"], ha="center", fontsize=11.5, fontweight="bold", color=H(INK))
    ax.set_xlim(-0.1, 3.3); ax.set_ylim(-0.6, 2.6)
    ax = axes[3]
    segs4 = [((0, 0), (1.5, 0)), ((1.5, 0), (3, 0)), ((1.5, 0), (1.5, 1.5)), ((1.5, -1.5), (1.5, 0))]
    for a_, b_ in segs4:
        ax.plot([a_[0], b_[0]], [a_[1], b_[1]], color="#C9CEC8", lw=6)
    mids = [((a_[0] + b_[0]) / 2, (a_[1] + b_[1]) / 2) for a_, b_ in segs4]
    for (x1, y1), lab, (tx, ty) in [(mids[1], "0°", (2.0, 0.17)), (mids[2], "90°", (0.72, 0.55)), (mids[3], "90°", (0.72, -0.72))]:
        ax.add_patch(FancyArrowPatch(mids[0], (x1, y1), connectionstyle="arc3,rad=0.0" if lab == "0°" else "arc3,rad=-0.2",
                                     arrowstyle="-|>", mutation_scale=14, color=H(FOREST), lw=1.8, shrinkA=8, shrinkB=8))
        ax.text(tx, ty, lab, fontsize=12, color=H(FOREST), fontweight="bold", ha="center")
    for x, y in mids:
        ax.add_patch(Circle((x, y), 0.14, color=H(DARK), zorder=5))
    ax.text(1.5, -2.05, T["ang_d"], ha="center", fontsize=11, color=H(INK))
    ax.set_xlim(-0.3, 3.3); ax.set_ylim(-2.3, 1.8)
    for ax, ttl in zip(axes, T["ang_titles"]):
        ax.set_title(ttl, loc="left"); ax.set_aspect("equal"); ax.set_axis_off(); ax.set_anchor("N")
    plt.tight_layout()
    fig.savefig(D / "concept_angular.png"); plt.close(fig)
    # appendix: N1 concept
    L1 = T["n1c"]
    fig, ax = plt.subplots(figsize=(7, 7))
    street_grid(ax, 7)
    ax.add_patch(Circle((1.5, 1.5), 4.6, fill=False, ls=(0, (5, 4)), ec=H(MUTED), lw=1.4))
    ax.text(4.75, 5.25, L1[4], fontsize=10, color=H(MUTED))
    route(ax, [(2, 1), (6, 1)], lw=3.2); route(ax, [(2, 2), (4, 2), (4, 4)], color="#F2A65A", lw=3.2)
    route(ax, [(1, 2), (1, 3), (2, 3), (2, 5)], color="#F7C99B", lw=3.2)
    park(ax, 1, 1, "A", color=H(DARK)); park(ax, 6, 0, "B"); park(ax, 4, 4, "C"); park(ax, 2, 5, "D"); park(ax, 6, 6, "E", color="#9AA39C")
    for x, y in [(4, 2), (1, 3), (2, 3)]:
        dot(ax, x, y, r=0.09)
    ax.text(4.0, 0.62, L1[0], fontsize=11, color=H(ACCENT), fontweight="bold", ha="center")
    ax.text(4.12, 3.2, L1[1], fontsize=11, color="#D9822B", fontweight="bold")
    ax.text(0.08, 4.25, L1[2], fontsize=11, color="#C98A4B", fontweight="bold")
    ax.text(6.5, 7.15, L1[3], fontsize=10, color=H(MUTED), ha="center")
    finish(ax, (-0.2, 7.3), (-0.3, 7.4))
    fig.savefig(D / "concept_N1.png"); plt.close(fig)
    # appendix: N2 concept (street-axis continuity)
    L2 = T["n2c"]
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    street_grid(ax, 8)
    stroke = [(0, 1), (2, 2), (4, 3), (6, 4.2), (8, 5.6)]
    ax.plot(*zip(*stroke), color=H(FOREST), lw=6, alpha=0.9, zorder=2, solid_capstyle="round")
    ax.plot([5, 5], [0, 8], color="#8C968E", lw=4, zorder=2)
    ax.text(8.1, 5.75, L2[0], color=H(FOREST), fontsize=11, fontweight="bold", va="bottom")
    ax.text(5.15, 7.6, L2[1], color="#6F7A72", fontsize=10, va="center")
    for (x, y, lab) in [(0.75, 1.55, "P1"), (2.6, 2.55, "P2"), (6.95, 4.95, "P3")]:
        ax.add_patch(Rectangle((x, y), 0.6, 0.55, color=H(MOSS), ec=H(FOREST), lw=1.5, zorder=4))
        ax.text(x + 0.3, y + 0.27, lab, color=H(DARK), ha="center", va="center", fontsize=10, fontweight="bold", zorder=5)
    ax.add_patch(Rectangle((5.15, 1.3), 0.6, 0.55, color="#9AA39C", zorder=4))
    ax.text(5.45, 1.57, "P4", color="white", ha="center", va="center", fontsize=10, fontweight="bold", zorder=5)
    for a_, b_ in [((1.05, 2.1), (2.9, 3.1)), ((2.9, 3.1), (7.25, 5.5))]:
        ax.add_patch(FancyArrowPatch(a_, b_, connectionstyle="arc3,rad=-0.35", arrowstyle="<->", color=H(ACCENT), lw=2, ls="--", zorder=6))
    ax.text(0.2, 4.4, L2[2], fontsize=11.5, color=H(ACCENT), fontweight="bold", bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none"), zorder=7)
    ax.text(6.4, 1.2, L2[3], fontsize=10.5, color=H(MUTED), bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none"), zorder=7)
    ins = ax.inset_axes([0.795, 0.03, 0.205, 0.36])
    ins.plot([0, 1], [0, 0], color=H(FOREST), lw=4); ins.plot([1, 2], [0, 0.35], color=H(FOREST), lw=4)
    ins.plot([1, 1.5], [0, -0.9], color="#B9C0B9", lw=3)
    ins.text(0.0, 0.55, L2[5], fontsize=9.5, color=H(FOREST)); ins.text(0.0, -1.35, L2[6], fontsize=9.5, color=H(MUTED))
    ins.set_xlim(-0.1, 2.1); ins.set_ylim(-1.6, 1.0); ins.set_axis_off(); ins.set_title(L2[4], fontsize=10)
    finish(ax, (-0.2, 10.2), (-0.3, 8.3))
    fig.savefig(D / "concept_N2.png"); plt.close(fig)
    # appendix: N3 toy (near vs far pair)
    L3 = T["n3c"]
    NG, RS3 = 14, 6
    SEGS = [((x, y), (x + 1, y)) for y in range(NG + 1) for x in range(NG)] + [((x, y), (x, y + 1)) for x in range(NG + 1) for y in range(NG)]

    def afield(px, py):
        f = set()
        for s in SEGS:
            (x0, y0), (x1, y1) = s
            if y0 == y1 and y0 in (py, py + 1) and max(x0, x1) <= px + 1 + RS3 and min(x0, x1) >= px - RS3:
                f.add(s)
            if x0 == x1 and x0 in (px, px + 1) and max(y0, y1) <= py + 1 + RS3 and min(y0, y1) >= py - RS3:
                f.add(s)
        return f

    def mfield(px, py):
        f = set()
        for s in SEGS:
            mx, my = (s[0][0] + s[1][0]) / 2, (s[0][1] + s[1][1]) / 2
            if max(px - mx, 0, mx - (px + 1)) + max(py - my, 0, my - (py + 1)) <= RS3 - 0.5:
                f.add(s)
        return f

    A_ = (1, 6)
    fig, axes = plt.subplots(2, 2, figsize=(10, 10.8), gridspec_kw=dict(hspace=0.32, wspace=0.08))
    toy = {}
    for row, (lab, B_) in enumerate([(L3[2], (4, 6)), (L3[3], (10, 6))]):
        f1, f2, m1, m2 = afield(*A_), afield(*B_), mfield(*A_), mfield(*B_)
        ja, jm = len(f1 & f2) / len(f1 | f2), len(m1 & m2) / len(m1 | m2)
        toy[row] = (ja, jm)
        for col, (fA, fB, ttl, J) in enumerate([(f1, f2, L3[0], ja), (m1, m2, L3[1], jm)]):
            ax = axes[row, col]
            for sg in SEGS:
                c, lw = "#E4E7E2", 0.8
                if sg in fA and sg in fB:
                    c, lw = "#6A3D9A", 4
                elif sg in fA:
                    c, lw = H(FOREST), 2.6
                elif sg in fB:
                    c, lw = H(ACCENT), 2.6
                ax.plot([sg[0][0], sg[1][0]], [sg[0][1], sg[1][1]], color=c, lw=lw, solid_capstyle="butt")
            park(ax, *A_, "A", color=H(DARK)); park(ax, *B_, "B", color=H(DARK))
            ax.set_title(f"{ttl}:  J = {J:.2f}", fontsize=13); ax.set_aspect("equal"); ax.set_axis_off()
        is_edge = ja >= 0.1 - 1e-9 and ja / jm >= 1.0
        fig.text(0.13, axes[row, 0].get_position().y1 + 0.045, f"{lab}:  J_ang = {ja:.2f},  {L3[4]} = {ja / jm:.2f}  →  {L3[5] if is_edge else L3[6]}",
                 fontsize=13.5, fontweight="bold", color=H(FOREST) if is_edge else H(MUTED))
    legend_items(axes[0, 0], list(zip(L3[7:10], [H(FOREST), H(ACCENT), "#6A3D9A"], ["-"] * 3)), loc="lower left", fontsize=10)
    fig.savefig(D / "concept_N3.png"); plt.close(fig)
    use_mpl_lang("en")
    return toy


TOY = {}
for lang in LANGS:
    TOY = make_lang_figs(lang)
    print(lang, sorted(p.name for p in (FIG / lang).iterdir()))

en ['D_overview.png', 'D_zoom.png', 'N1_overview.png', 'N1_zoom.png', 'N2_overview.png', 'N2_zoom.png', 'N3_overview.png', 'N3_zoom.png', 'angular_field.png', 'attachment.png', 'concept_N1.png', 'concept_N2.png', 'concept_N3.png', 'concept_angular.png', 'cross_status.png', 'network_S_vs_C.png', 'rings.png', 'walk_split.png']


ko ['D_overview.png', 'D_zoom.png', 'N1_overview.png', 'N1_zoom.png', 'N2_overview.png', 'N2_zoom.png', 'N3_overview.png', 'N3_zoom.png', 'angular_field.png', 'attachment.png', 'concept_N1.png', 'concept_N2.png', 'concept_N3.png', 'concept_angular.png', 'cross_status.png', 'network_S_vs_C.png', 'rings.png', 'walk_split.png']


## 12-7. 슬라이드 문구 (영어 / 한국어)

In [9]:
pct = lambda v: f"{v * 100:.0f}%"
S = {
"en": dict(
  title=["How Are New York's Parks", "Connected by Streets?"],
  subtitle=f"Grouping {N['main']:,} small and medium parks by the shape of the walking network — not just by distance",
  footer=["NYC Parks properties · OpenStreetMap walking network · Cityseer 5.8", time.strftime("%B %Y")],
  s2_title="Close is not the same as connected",
  s2_why="Why: next, we test whether connected groups of parks go with more shops and visits nearby.",
  s2_body=["We usually ask how far apart two parks are. But people pick routes that are easy to follow.",
           "A route with few turns feels simple — like walking along one straight avenue.",
           "So we ask: which parks are linked by the street layout itself?"],
  s2_note="This idea comes from Space Syntax, a field that studies how the shape of streets guides where people walk.",
  s2_caps=["Close on the map — but you must walk around a barrier", "Farther apart — but one straight street"],
  s3_title="Four ways two parks can be “connected”",
  s3_defs=[("D", "Short walk", "The two parks are only a few minutes' walk apart along real streets (200–800 m).", "The usual “nearby” — our baseline."),
           ("N1", "Few turns", "You can walk from one park to the other while turning very little.", "Directions that say: “just go straight.”"),
           ("N2", "Same street axis", "Both parks sit along one continuous street you can follow without switching streets.", "Two shops on the same avenue."),
           ("N3", "Shared walking zone", "The streets reached by walking (almost) straight from each park overlap more than distance explains.", "Two schools whose students use the same streets.")],
  s3_rule="For all four: every park in a group must be within a 10-minute walk (800 m) of every other park in that group.",
  s4_title="What we used",
  s4_stats=[(f"{N['main']:,}", "small & medium parks — we left out huge parks such as Central Park, and lots that are not real parks"),
            (f"{N['segC'] / 1000:,.0f}k", "walkable streets and paths from OpenStreetMap, the free “Wikipedia of maps”"),
            (f"{N['split']} → {N['pieces']}", "very long parks (e.g. parkway medians) cut into ~400 m pieces, so one park cannot link places kilometres apart")],
  s4_cap="Walking network (lines) and parks (green) around Fort Greene, Brooklyn",
  s5_title="Counting how much a route turns",
  s5_body=["Going straight = 0°. Turning a corner ≈ 90°.",
           "Between two parks we find the route that turns the least (not the shortest one) and add up its turns.",
           "Map: from one park, dark-blue streets are reached by going straight, light blue with one turn, orange with two.",
           "Computed with Cityseer, an open-source tool for analysing street networks."],
  s6_title="Where does a park meet the street?",
  s6_body=["We don't know where each park's entrances are.",
           "So we treat the streets running along the park's edges (orange) as its doors.",
           f"Big parks have many edge streets, so we start from at most {K_SOURCES} spread-out ones (black). This keeps big parks from getting an unfair advantage."],
  sD_title="D · Parks within a short walk (baseline)",
  sD_stats=[(f"{int(FD.groups)}", "groups"), (f"{int(FD.parks_in_groups):,}", "parks in a group"),
            (f"{int(FD.size_median)}", "parks in a typical group"), (f"{int(FD.robust_isolated)}", "parks with no other park nearby")],
  sD_cap="Each colour is one group. This is pure proximity — the other three definitions are compared against it.",
  sD_note="D links parks whose shortest walking distance is ≤ 200/400/800 m; same grouping procedure as the others.",
  s7_title="N1 · Parks linked by few-turn routes",
  s7_stats=[(f"{int(F2.groups)}", "walkable groups"), (f"{int(F2.parks_in_groups):,}", "parks in a group"),
            (f"{int(F2.size_median)}", "parks in a typical group"), (f"{int(F2.robust_isolated)}", "parks with no few-turn partner")],
  s7_cap="Each colour is one group. Orange lines: few-turn links within a 10-minute walk.",
  s8_title="N2 · Parks along the same street axis",
  s8_stats=[(f"{int(FN2.groups)}", "groups"), (f"{int(FN2.parks_in_groups):,}", "parks in a group")],
  s8_body=["“Street axis” = one continuous street you can follow without switching streets (direction change < 30° at each junction).",
           f"{pct(MAIN_N2)} of N2 groups touch a main street (top 5% of through-walking) — kept as a group attribute."],
  s9_title="N3 · Parks sharing a walking zone",
  s9_stats=[(f"{int(FN3.groups)}", "groups (mostly pairs)"), (f"{int(FN3.parks_in_groups)}", "parks in a group")],
  s9_body=["Rare: two parks' straight-walk zones seldom overlap more than their distance alone explains.",
           "N3 is a weak signal — useful mainly as a contrast to N1 and N2."],
  s10_title="Groups must be walkable: the 10-minute rule",
  s10_body=["Street-shape links can chain parks across several kilometres.",
            "So inside each group, every pair of parks must be within a 10-minute walk (800 m along real streets).",
            "Long groups are split; a park with nobody within 10 minutes leaves the group."],
  s10_head=["", "groups", "largest group"],
  s10_rows=["D short walk", "N1 few turns", "N2 same axis", "N3 walking zone"],
  s11_title="Is this just about distance?",
  s11_cats=["N1 few turns", "N2 same axis", "N3 walking zone"],
  s11_series="grouped park pairs that distance alone (D) also groups",
  s11_body=["We compare each definition with D, which groups parks by walking distance only.",
            f"N1: {pct(ALSO_D['N1'])} of its grouped pairs are also grouped by D — the other {pct(1 - ALSO_D['N1'])} come from street shape.",
            f"N2: {pct(ALSO_D['N2'])} · N3: {pct(ALSO_D['N3'])}.",
            "So street shape changes which nearby parks belong together."],
  s12_title="Parks left alone",
  s12_chart=f"Isolated under all three definitions ({N['iso_all']} parks)",
  s12_body=["Mostly in Staten Island and Queens, where streets curve and dead-end.",
            "Also parks squeezed next to highways or along the waterfront.",
            "They sit outside the walkable “park network” — worth a look for planning."],
  s14_title="Next step: parks and commercial activity",
  s14_body=["Every park belongs to one cluster per definition (a single park counts as its own cluster).",
            f"Each cluster gets a {COMMERCIAL_BUFFER} m ring around its parks — its commercial area.",
            "Next: count shops, business variety and visits in each ring (Advan POIs) and compare clusters across D, N1, N2 and N3, controlling for park size, population, jobs and street accessibility."],
  s14_card="Ready now: cluster tables with park area, walking span, internal links, main-street share and isolation.",
  s14_note="Outputs in outputs/NYC/clusters: park_cluster_membership.csv, clusters_<def>.csv/.geojson (100 m commercial rings).",
  s13_title="Takeaways",
  s13_cards=[("Street shape matters", f"Few-turn links form {int(F2.groups)} walkable park groups across NYC; {int(F2.robust_isolated)} parks have no few-turn partner."),
             ("Not just distance", f"{pct(1 - ALSO_D['N1'])} of few-turn group pairs are not grouped by walking distance alone."),
             ("Four lenses", "Distance, few turns, same street axis and shared zone give different groups — ready for the commercial analysis.")],
  s13_next="Next: measure shops and visits in each cluster's 100 m ring and run the regressions.",
  app_title="Appendix · Technical details",
  app_items=["Pipeline and tools", "Network representation and park attachment", "Angular cost on the dual graph",
             "D, N1, N2, N3: formal definitions and parameters", "From park graphs to groups (percolation)", "Proximity check", "Limitations"],
  ad_title="D · Walking distance (formal)",
  ad_eq="edge  ⇔  D_ij ≤ d",
  ad_eq_note="D = shortest walking distance between the two parks' attachment segments (Cityseer shortest-path trees)",
  ad_body=["3 settings: d ∈ {200, 400, 800} m", "Includes adjacent parks (shared frontage, touching polygons) — proximity itself",
           "Same grouping procedure as N1–N3 (percolation filter, Louvain γ = 2, consensus, 10-min split): the proximity baseline for RQ3"],
  a1_title="Pipeline and tools",
  a1_steps=["OSM pedestrian network (C / S)", "Cityseer dual graph", "Park attachment (no entrances)",
            "Angular relations & centrality", "Park graphs N1 · N2 · N3", "Persistence, consensus & 10-min split"],
  a1_body=[f"Parks: NYC Parks properties ({N['raw']:,}) → flagship parks, non-park parcels and parks > 100 acres removed → {N['main']:,} parks; "
           f"{N['split']} long linear parks split into {N['pieces']} pieces ({N['features']:,} graph nodes)",
           "Network: OSM via OSMnx with a custom walk filter (keeps streets tagged sidewalk=separate), NYC + 3.2 km buffer; "
           "borough networks with 3.2 km non-reporting buffers",
           "Software: Cityseer 5.8 (dual graph, simplest/shortest-path Dijkstra, angular centrality), NetworkX (graphs, Louvain), GeoPandas/Shapely"],
  a2_title="Network representation and park attachment",
  a2_body=["S draws sidewalks and crossings separately: parks facing each other across a street are ~180° apart.",
           "C (street centrelines + pedestrian-only paths) behaves like a Space Syntax segment map → main analysis; S = sensitivity.",
           f"Attachment: segments within ±{ATTACH_BUFFER['C']:.0f} m of the boundary whose in-band length ≥ 50% of the segment or ≥ min(2.5·d, 25% of perimeter)",
           f"K = {K_SOURCES} sources by farthest-point sampling; “unresolved” if attached only to components < {SMALL_COMPONENT} segments"],
  a3_title="Angular cost on the dual graph",
  a3_body=[("Simplest path  ", "Cityseer routes on the dual graph (street segments = nodes) and minimises the cumulative change of direction in degrees, including bends inside segments."),
           ("Metric bound  ", "Each search runs up to 3.2 km; R ∈ {800, 1600, 3200} m filters the path length afterwards — distance bounds the search, it does not define the edge.")],
  a4_title="N1 · Formal definition",
  a4_eq=["A_ij(R) = min over source u ∈ B_i, target v ∈ B_j of angular cost(u → v),  path length ≤ R", "edge  ⇔  A_ij ≤ θ"],
  a4_table=[("θ", "reading", "edges from A"), ("15°", "same straight street", "B"), ("45°", "nearly straight", "B"),
            ("90°", "one right-angle turn (strict)", "B, C"), ("180°", "two turns", "B, C, D")],
  a4_body=["24 settings: θ ∈ {15, 45, 90, 180}° × R ∈ {800, 1600, 3200} m × {best pair, ≥ 2 robust pairs}",
           f"Groups: Louvain communities on the {len(KEEP)} non-percolating settings (weights exp(−A/45°)), consensus ≥ 0.5, then split to a 10-min walking diameter",
           "Measured one-turn routes cost ~90–100° (bends inside segments), so θ = 90° is strict"],
  a5_title="N2 · Street-axis continuity (formal)",
  a5_eq="edge  ⇔  same stroke and gap_along ≤ lim",
  a5_eq_note="stroke = segments chained where the direction changes < 30° at a junction (20°, 45° also tested)",
  a5_body=["Gap: distance along the stroke between the two parks' attachment intervals (0 if they overlap)",
           "9 settings: stroke angle ∈ {20, 30, 45}° × lim ∈ {400, 800, 1600} m; E0 pairs excluded; network C",
           "Main street = stroke in the top 5% / 10% of network length by mean NACH-1600, NACH_r = log(choice_r + 1) / log(depth_r + 3) — a cluster attribute, not an edge condition",
           "Groups: same procedure as all definitions, split to a 10-min walking diameter"],
  a6_title="N3 · Formal definition",
  a6_body=["Field = segments reachable with angular cost ≤ θc and path length ≤ R",
           "J = length-weighted Jaccard of two parks' fields, for angular and metric fields",
           "Edge if J_ang ≥ τ and J_ang / J_met ≥ κ",
           "16 settings: θc ∈ {45, 90}° × R ∈ {800, 1600} m × τ ∈ {0.1, 0.2} × κ ∈ {1.0, 1.25}"],
  a6_cap=f"Toy grid: the near pair shares fields only because it is close (ratio {TOY[0][0] / TOY[0][1]:.2f}); the far pair on one street keeps sharing its few-turn field (ratio {TOY[1][0] / TOY[1][1]:.2f}).",
  a7_title="From park graphs to robust groups",
  a7_chart="N1: largest connected component (share of a borough's parks, max over boroughs)",
  a7_series="R = {R} m",
  a7_body=[("Percolation  ", "Once θ ≥ 45° at R ≥ 1.6 km, or one turn is allowed, most of a borough's parks fall into one component; Queens and Staten Island percolate much later."),
           ("Communities  ", "In each setting, Louvain communities find internally dense sets of parks (resolution γ = 1, 2, 4)."),
           ("Consensus  ", "Co-membership share across settings → average-linkage clusters (≥ 0.5) → split to a 10-min walking diameter."),
           ("Isolation  ", "IsolationPersistence = share of settings with no configurational edge; ≥ 0.8 = robust isolated.")],
  a8_title="Proximity check (AUC)",
  a8_cats=["N1 few turns", "N1rel (relative)", "N2 same axis", "N3 walking zone"],
  a8_series=["Within the search bound R (verdict)", "All pairs ≤ 3.2 km"],
  a8_body=[("Proximity AUC: ", "probability that a connected pair is closer than an unconnected one; > 0.85 = proximity-dominated."),
           ("Result: ", "no definition crosses 0.85 once distance is held to its search bound."),
           ("Spearman ", "(angular cost vs network distance) ≈ 0.55.")],
  a9_title="Limitations",
  a9_body=["Percolation: allowing one turn connects almost all parks in a borough — groups rely on strict settings and community detection",
           "Turn costs include bends inside segments (one turn ≈ 90–100°), so θ = 90° is a strict one-turn threshold",
           "OSM completeness and mapping style (separately drawn sidewalks) affect angular costs; S results differ from C",
           "Park entrances unknown (boundary attachment); N2 computed on C only; long linear parks split into ~400 m pieces",
           f"D: only the 200 m setting avoids percolation, so D groups are tight walking-distance groups",
           f"N1 and N2 groups overlap substantially (ARI {ARI_N1N2:.2f}) — street-axis continuity is a stricter subset of few-turn links",
           "Groups not yet validated against observed park use or pedestrian counts"],
  notes=["Space-Syntax-inspired analysis of NYC parks with Cityseer. Map colours = few-turn (N1) groups.",
         "Contrast: proximity vs configuration.",
         "One-page summary of the four definitions (D, N1, N2, N3); formal versions are in the appendix.",
         "Data sources and filters.", "Angular (simplest-path) cost explained.", "Boundary-based attachment, no entrance data.",
         "N1 results.", "N2 results.", "N3 results.", "Walking-distance safeguard.", "Share of grouped pairs also grouped by D.",
         "Isolated under N1, N2 and N3.", "Summary.", "Appendix."],
),
"ko": dict(
  title=["뉴욕의 공원들은", "길로 어떻게 이어져 있을까?"],
  subtitle=f"거리만이 아니라 ‘길의 모양’으로 중소형 공원 {N['main']:,}곳의 묶음 찾기",
  footer=["NYC 공원 자료 · 오픈스트리트맵 보행망 · Cityseer 5.8", time.strftime("%Y년 %m월")],
  s2_title="가깝다고 다 이어진 것은 아니다",
  s2_why="왜: 다음 단계에서 이어진 공원 묶음 주변에 상점과 방문이 더 많은지 검증합니다.",
  s2_body=["보통은 두 공원이 얼마나 떨어져 있는지를 봅니다. 하지만 사람은 ‘따라가기 쉬운 길’을 고릅니다.",
           "덜 꺾이는 길은 단순하고 찾기 쉽습니다. 큰길 하나를 따라 쭉 걷는 것처럼요.",
           "그래서 이렇게 묻습니다: 길의 짜임새 자체가 이어 주는 공원은 어디일까?"],
  s2_note="이 생각은 길의 모양이 사람의 움직임을 어떻게 이끄는지 연구하는 ‘스페이스 신택스(Space Syntax)’에서 왔습니다.",
  s2_caps=["지도에선 가깝지만, 장애물을 빙 돌아가야 함", "더 멀지만, 곧은 길 하나로 쭉"],
  s3_title="공원 두 곳이 ‘이어져 있다’는 네 가지 뜻",
  s3_defs=[("D", "가까운 거리", "실제 길로 걸어서 몇 분(200–800 m) 거리 안에 있다.", "흔히 말하는 ‘가깝다’ — 비교 기준선"),
           ("N1", "적게 꺾는 길", "한 공원에서 다른 공원까지 거의 꺾지 않고 걸어갈 수 있다.", "“쭉 직진하세요” 한마디로 끝나는 길 안내"),
           ("N2", "같은 길(축) 위", "두 공원이 다른 길로 갈아타지 않고 쭉 따라갈 수 있는 하나의 길 위에 있다.", "같은 대로변에 있는 두 가게"),
           ("N3", "겹치는 생활권", "각 공원에서 거의 직진만으로 닿는 길들이 거리만으로 기대되는 것보다 많이 겹친다.", "같은 골목을 쓰는 두 학교")],
  s3_rule="네 정의 모두: 한 묶음 안의 공원끼리는 서로 걸어서 10분(800 m) 이내여야 합니다.",
  s4_title="사용한 자료",
  s4_stats=[(f"{N['main']:,}", "중소형 공원 — 센트럴파크 같은 초대형 공원과 공원이 아닌 부지는 제외"),
            (f"{N['segC'] / 1000:,.0f}k", "걸을 수 있는 길·산책로 (무료 지도 ‘오픈스트리트맵’, 지도판 위키백과)"),
            (f"{N['split']} → {N['pieces']}", "아주 긴 공원(도로 중앙 녹지 등)은 약 400 m씩 잘라, 공원 하나가 몇 km 떨어진 곳을 잇지 못하게 함")],
  s4_cap="브루클린 포트그린 주변의 보행망(선)과 공원(초록)",
  s5_title="길이 얼마나 꺾이는지 세기",
  s5_body=["직진은 0°, 모퉁이 하나를 돌면 약 90°.",
           "두 공원 사이에서 ‘가장 짧은 길’이 아니라 ‘가장 덜 꺾는 길’을 찾아, 꺾인 각도를 모두 더합니다.",
           "지도: 한 공원에서 출발해 진한 파랑 길은 직진만으로, 연한 파랑은 한 번, 주황은 두 번 꺾어야 닿습니다.",
           "계산은 길 네트워크 분석용 오픈소스 도구 Cityseer로 했습니다."],
  s6_title="공원은 어디서 길과 만날까?",
  s6_body=["공원 입구가 어디인지 알려 주는 자료가 없습니다.",
           "그래서 공원 가장자리를 따라 난 길(주황)을 공원의 ‘문’으로 봅니다.",
           f"큰 공원은 가장자리 길이 많아 유리해지므로, 고르게 흩어진 최대 {K_SOURCES}곳(검정)에서만 출발합니다."],
  sD_title="D · 걸어서 가까운 공원들 (기준선)",
  sD_stats=[(f"{int(FD.groups)}", "묶음"), (f"{int(FD.parks_in_groups):,}", "묶음에 속한 공원"),
            (f"{int(FD.size_median)}", "보통 묶음의 공원 수"), (f"{int(FD.robust_isolated)}", "가까운 공원이 없는 공원")],
  sD_cap="색 하나 = 묶음 하나. 순수한 ‘가까움’만으로 만든 묶음 — 나머지 세 정의는 이것과 비교합니다.",
  sD_note="D는 보행 최단거리가 200/400/800 m 이하인 공원을 잇고, 다른 정의와 같은 묶음 절차를 씁니다.",
  s7_title="N1 · 적게 꺾는 길로 이어진 공원들",
  s7_stats=[(f"{int(F2.groups)}", "걸어갈 수 있는 묶음"), (f"{int(F2.parks_in_groups):,}", "묶음에 속한 공원"),
            (f"{int(F2.size_median)}", "보통 묶음의 공원 수"), (f"{int(F2.robust_isolated)}", "적게 꺾는 짝이 없는 공원")],
  s7_cap="색 하나 = 묶음 하나. 주황 선: 걸어서 10분 이내의 적게 꺾는 연결.",
  s8_title="N2 · 같은 길(축) 위의 공원들",
  s8_stats=[(f"{int(FN2.groups)}", "묶음"), (f"{int(FN2.parks_in_groups):,}", "묶음에 속한 공원")],
  s8_body=["‘길 축’ = 다른 길로 갈아타지 않고 쭉 따라갈 수 있는 연속된 길 (교차점마다 방향 변화 30° 미만).",
           f"N2 묶음의 {pct(MAIN_N2)}가 큰길(통과 보행 상위 5%)에 닿아 있습니다 — 묶음 속성으로 기록."],
  s9_title="N3 · 생활권을 공유하는 공원들",
  s9_stats=[(f"{int(FN3.groups)}", "묶음 (대부분 2곳씩)"), (f"{int(FN3.parks_in_groups)}", "묶음에 속한 공원")],
  s9_body=["드뭅니다: 두 공원의 생활권이 ‘가까워서’ 겹치는 정도보다 더 많이 겹치는 경우는 적습니다.",
           "N3는 약한 신호 — 주로 N1·N2와 비교하는 용도입니다."],
  s10_title="묶음은 걸어갈 수 있어야: 10분 규칙",
  s10_body=["길 모양으로만 이으면 공원들이 몇 km씩 사슬처럼 이어질 수 있습니다.",
            "그래서 묶음 안의 모든 공원 쌍은 실제 길로 걸어서 10분(800 m) 이내여야 합니다.",
            "너무 긴 묶음은 나누고, 10분 안에 짝이 없는 공원은 묶음에서 빠집니다."],
  s10_head=["", "묶음 수", "가장 큰 묶음"],
  s10_rows=["D 가까운 거리", "N1 적게 꺾는 길", "N2 같은 길(축)", "N3 겹치는 생활권"],
  s11_title="결국 거리 문제 아닐까?",
  s11_cats=["N1 적게 꺾는 길", "N2 같은 길(축)", "N3 겹치는 생활권"],
  s11_series="거리(D)로도 같은 묶음이 되는 공원 쌍의 비율",
  s11_body=["각 정의를 걸어서 가까운지만 보는 D와 비교합니다.",
            f"N1: 묶인 공원 쌍의 {pct(ALSO_D['N1'])}는 D에서도 묶임 — 나머지 {pct(1 - ALSO_D['N1'])}는 길 모양 때문에 묶임.",
            f"N2: {pct(ALSO_D['N2'])} · N3: {pct(ALSO_D['N3'])}.",
            "즉, 길의 모양이 ‘가까운 공원들 중 누가 함께 묶이는지’를 바꿉니다."],
  s12_title="홀로 떨어진 공원들",
  s12_chart=f"세 정의 모두에서 고립된 공원 ({N['iso_all']}곳)",
  s12_body=["주로 길이 굽고 막다른 길이 많은 스태튼아일랜드와 퀸스에 있습니다.",
            "고속도로 옆이나 물가에 끼인 공원도 많습니다.",
            "걸어서 이어지는 ‘공원 네트워크’에서 빠진 곳 — 계획할 때 살펴볼 만합니다."],
  s14_title="다음 단계: 공원과 상업 활동",
  s14_body=["모든 공원은 정의마다 하나의 군집에 속합니다 (단독 공원은 그 자체로 하나의 군집).",
            f"각 군집의 공원 주변 {COMMERCIAL_BUFFER} m를 그 군집의 상업 영역으로 봅니다.",
            "다음: 공원 규모·인구·일자리·가로 접근성을 통제한 뒤, 영역 안의 상점 수·업종 다양성·방문량(Advan POI)을 D·N1·N2·N3 군집끼리 비교합니다."],
  s14_card="준비 완료: 공원 면적, 보행 범위, 내부 연결, 큰길 비율, 고립 지수를 담은 군집 테이블.",
  s14_note="산출물: outputs/NYC/clusters 의 park_cluster_membership.csv, clusters_<def>.csv/.geojson (100 m 상업 영역).",
  s13_title="정리",
  s13_cards=[("길의 모양이 중요", f"적게 꺾는 길 기준으로 걸어갈 수 있는 공원 묶음 {int(F2.groups)}개, 짝이 없는 공원 {int(F2.robust_isolated)}곳."),
             ("거리만의 문제가 아님", f"적게 꺾는 길로 묶인 공원 쌍의 {pct(1 - ALSO_D['N1'])}는 거리만으로는 묶이지 않습니다."),
             ("네 가지 렌즈", "거리, 적게 꺾는 길, 같은 길(축), 겹치는 생활권은 서로 다른 묶음을 만듭니다 — 상업 분석 준비 완료.")],
  s13_next="다음 단계: 각 군집의 100 m 영역에서 상점과 방문을 재고 회귀분석하기.",
  app_title="부록 · 기술적 세부사항",
  app_items=["분석 파이프라인과 도구", "보행망 표현과 공원 연결 규칙", "듀얼 그래프 위의 각도 비용",
             "D·N1·N2·N3 형식 정의와 파라미터", "공원 그래프에서 묶음으로 (퍼콜레이션)", "근접성 검증", "한계"],
  ad_title="D · 보행거리 (형식적 정의)",
  ad_eq="연결  ⇔  D_ij ≤ d",
  ad_eq_note="D = 두 공원 연결 세그먼트 사이의 보행 최단거리 (Cityseer 최단경로 트리)",
  ad_body=["3개 설정: d ∈ {200, 400, 800} m", "맞닿은 공원(같은 가로 세그먼트, 폴리곤 접촉) 포함 — 근접성 그 자체",
           "N1–N3와 같은 묶음 절차(퍼콜레이션 필터, Louvain γ = 2, 합의, 10분 분할) — RQ3의 근접성 기준선"],
  a1_title="분석 파이프라인과 도구",
  a1_steps=["OSM 보행망 (C / S)", "Cityseer 듀얼 그래프", "공원-네트워크 연결 (입구 없음)",
            "각도 관계·중심성", "공원 그래프 N1·N2·N3", "지속성·합의·10분 분할"],
  a1_body=[f"공원: NYC 공원 자료 {N['raw']:,}곳 → Flagship·비공원 부지·100 acres 초과 제외 → {N['main']:,}곳; "
           f"긴 선형 공원 {N['split']}곳을 {N['pieces']}개 구간으로 분할 (그래프 노드 {N['features']:,}개)",
           "보행망: OSMnx로 OSM 수집, sidewalk=separate 도로를 살린 맞춤 walk 필터, NYC + 3.2 km 버퍼; 보로별 네트워크 + 3.2 km 비보고 버퍼",
           "도구: Cityseer 5.8 (듀얼 그래프, simplest/shortest Dijkstra, 각도 중심성), NetworkX (그래프, Louvain), GeoPandas/Shapely"],
  a2_title="보행망 표현과 공원 연결 규칙",
  a2_body=["S는 보도·횡단보도를 따로 그림: 길 건너 마주 보는 공원도 약 180° 떨어진 것으로 계산됨.",
           "C(차도 중심선 + 보행 전용로)는 Space Syntax 세그먼트 맵과 비슷 → 주 분석, S는 민감도 분석.",
           f"연결: 경계 ±{ATTACH_BUFFER['C']:.0f} m 띠 안 길이가 세그먼트의 50% 이상이거나 min(2.5·d, 둘레의 25%) 이상인 세그먼트",
           f"출발점 K = {K_SOURCES} (farthest-point sampling); {SMALL_COMPONENT} 세그먼트 미만 연결요소에만 붙으면 ‘판정 불가’"],
  a3_title="듀얼 그래프 위의 각도 비용",
  a3_body=[("Simplest path  ", "Cityseer는 듀얼 그래프(길 세그먼트 = 노드)에서 누적 방향 변화(°)를 최소화하는 경로를 찾음. 세그먼트 안의 굴곡도 포함."),
           ("거리 한계  ", "탐색은 최대 3.2 km; R ∈ {800, 1600, 3200} m는 경로 길이를 사후 필터링 — 거리는 탐색 범위일 뿐 연결의 정의가 아님.")],
  a4_title="N1 · 형식적 정의",
  a4_eq=["A_ij(R) = min (출발 u ∈ B_i, 도착 v ∈ B_j) 각도 비용(u → v),  경로 길이 ≤ R", "연결  ⇔  A_ij ≤ θ"],
  a4_table=[("θ", "의미", "A와 연결"), ("15°", "같은 곧은 길", "B"), ("45°", "거의 직진", "B"),
            ("90°", "직각 한 번 (엄격)", "B, C"), ("180°", "두 번 꺾음", "B, C, D")],
  a4_body=["24개 설정: θ ∈ {15, 45, 90, 180}° × R ∈ {800, 1600, 3200} m × {최적 쌍, 강건 쌍 ≥ 2}",
           f"묶음: 퍼콜레이션 없는 {len(KEEP)}개 설정에서 Louvain 커뮤니티(가중치 exp(−A/45°)), 합의 ≥ 0.5, 이후 도보 10분 지름으로 분할",
           "실측 한 번 꺾는 경로는 약 90–100°(세그먼트 내부 굴곡 포함) → θ = 90°는 엄격한 기준"],
  a5_title="N2 · 가로축 연속성 (형식적 정의)",
  a5_eq="연결  ⇔  같은 stroke 그리고 gap_along ≤ lim",
  a5_eq_note="stroke = 교차점에서 방향 변화 < 30°인 세그먼트를 이어 붙인 연속 가로 (20°, 45°도 시험)",
  a5_body=["간격: stroke를 따라 잰 두 공원 연결 구간 사이 거리 (겹치면 0)",
           "9개 설정: stroke 각도 ∈ {20, 30, 45}° × lim ∈ {400, 800, 1600} m; E0 쌍 제외; C 망",
           "큰길 = 평균 NACH-1600 기준 네트워크 연장 상위 5%/10% stroke, NACH_r = log(choice_r + 1) / log(depth_r + 3) — 연결 조건이 아닌 군집 속성",
           "묶음: 모든 정의와 같은 절차, 도보 10분 지름으로 분할"],
  a6_title="N3 · 형식적 정의",
  a6_body=["생활권 = 각도 비용 ≤ θc, 경로 길이 ≤ R로 닿는 세그먼트",
           "J = 두 공원 생활권의 길이 가중 Jaccard (각도 기준, 거리 기준 각각)",
           "연결 조건: J_ang ≥ τ 그리고 J_ang / J_met ≥ κ",
           "16개 설정: θc ∈ {45, 90}° × R ∈ {800, 1600} m × τ ∈ {0.1, 0.2} × κ ∈ {1.0, 1.25}"],
  a6_cap=f"예시 격자: 가까운 쌍은 가까워서 겹칠 뿐(비율 {TOY[0][0] / TOY[0][1]:.2f}), 같은 길 위의 먼 쌍은 적게 꺾는 생활권을 계속 공유(비율 {TOY[1][0] / TOY[1][1]:.2f}).",
  a7_title="공원 그래프에서 강건한 묶음으로",
  a7_chart="N1: 최대 연결요소 비율 (보로 공원 중 비율, 보로별 최댓값)",
  a7_series="R = {R} m",
  a7_body=[("퍼콜레이션  ", "θ ≥ 45°이고 R ≥ 1.6 km이거나 한 번 꺾기를 허용하면 보로 공원 대부분이 하나로 이어짐; 퀸스·스태튼아일랜드는 훨씬 늦게 이어짐."),
           ("커뮤니티  ", "설정마다 Louvain 커뮤니티로 내부가 촘촘한 공원 집합을 찾음 (해상도 γ = 1, 2, 4)."),
           ("합의  ", "설정 간 동반소속 비율 → 평균 연결 군집(≥ 0.5) → 도보 10분 지름으로 분할."),
           ("고립  ", "IsolationPersistence = 연결이 없는 설정의 비율; 0.8 이상 = 강건 고립.")],
  a8_title="근접성 검증 (AUC)",
  a8_cats=["N1 적게 꺾는 길", "N1rel (상대 기준)", "N2 같은 길(축)", "N3 겹치는 생활권"],
  a8_series=["탐색 한계 R 안 (판정 기준)", "3.2 km 안 모든 쌍"],
  a8_body=[("근접성 AUC: ", "연결된 쌍이 연결되지 않은 쌍보다 가까울 확률; 0.85 초과 = 근접성 지배."),
           ("결과: ", "거리를 탐색 한계 안으로 고정하면 어떤 정의도 0.85를 넘지 않음."),
           ("Spearman ", "(각도 비용 vs 네트워크 거리) ≈ 0.55.")],
  a9_title="한계",
  a9_body=["퍼콜레이션: 한 번 꺾기를 허용하면 보로 공원 대부분이 이어짐 — 묶음은 엄격한 설정과 커뮤니티 탐지에 의존",
           "꺾임 비용에 세그먼트 내부 굴곡 포함(한 번 꺾음 ≈ 90–100°) → θ = 90°는 엄격한 기준",
           "OSM 완전성과 표기 방식(보도 별도 표기)이 각도 비용에 영향; S 결과는 C와 다름",
           "공원 입구 미상(경계 기반 연결); N2는 C 망만 계산; 긴 선형 공원은 약 400 m 구간으로 분할",
           "D: 200 m 설정만 퍼콜레이션을 피함 → D 묶음은 아주 가까운 공원끼리의 묶음",
           f"N1과 N2 묶음이 꽤 비슷함(ARI {ARI_N1N2:.2f}) — 같은 길(축) 연결은 적게 꺾는 연결의 더 엄격한 부분집합에 가까움",
           "실제 공원 이용·보행량으로 묶음을 아직 검증하지 않음"],
  notes=["Cityseer를 이용한 Space Syntax 기반 NYC 공원 분석. 지도 색 = 적게 꺾는 길(N1) 묶음.",
         "근접성과 길의 짜임새 대비.", "네 정의(D·N1·N2·N3) 한 장 요약; 형식 정의는 부록.", "자료와 필터.", "각도(simplest path) 비용 설명.",
         "입구 자료 없이 경계 기반 연결.", "N1 결과.", "N2 결과.", "N3 결과.", "도보 거리 안전장치.",
         "묶인 공원 쌍 중 D에서도 묶이는 비율.", "N1·N2·N3 모두에서 고립.", "정리.", "부록."],
),
}

## 12-8. 덱 조립 도우미 (python-pptx, 언어별 글꼴)

In [10]:
CUR = {"lang": "en"}


def rgb(c):
    return RGBColor.from_string(c)


def style_run(r, size=16, color=INK, bold=False, italic=False, font=None, sub=False):
    f = r.font
    if CUR["lang"] == "ko":
        f.name = KO_FONT
    else:
        f.name = font or "Calibri"
    f.size = Pt(size)
    f.bold = bold
    f.italic = italic
    f.color.rgb = rgb(color)
    rPr = r._r.get_or_add_rPr()
    if CUR["lang"] == "ko":
        rPr.set("lang", "ko-KR")
        rPr.set("altLang", "en-US")
        latin = rPr.find(qn("a:latin"))
        ea = rPr.find(qn("a:ea"))
        if ea is None:
            ea = etree.SubElement(rPr, qn("a:ea"))
            latin.addnext(ea)
        ea.set("typeface", KO_FONT)
    if sub:
        rPr.set("baseline", "-25000")


def set_bullet(p, indent=0.25):
    pPr = p._p.get_or_add_pPr()
    pPr.set("marL", str(int(Inches(indent))))
    pPr.set("indent", str(-int(Inches(indent))))
    bf = etree.SubElement(pPr, qn("a:buFont"))
    bf.set("typeface", "Arial")
    bc = etree.SubElement(pPr, qn("a:buChar"))
    bc.set("char", "•")


def text(slide, x, y, w, h, paras, size=16, color=INK, font=None, bold=False, align=PP_ALIGN.LEFT,
         anchor=MSO_ANCHOR.TOP, bullets=False, space=6, line=1.1):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame
    tf.word_wrap = True
    tf.vertical_anchor = anchor
    tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
    for i, para in enumerate(paras):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.space_after = Pt(space)
        p.line_spacing = line
        for t, o in (para if isinstance(para, list) else [(para, {})]):
            r = p.add_run()
            r.text = t
            style_run(r, size=o.get("size", size), color=o.get("color", color), bold=o.get("bold", bold),
                      italic=o.get("italic", False), font=o.get("font", font), sub=o.get("sub", False))
        if bullets:
            set_bullet(p)
    return tb


def title(slide, t, sub=None, dark=False):
    text(slide, 0.6, 0.38, 12.1, 0.8, [t], size=30 if CUR["lang"] == "en" else 28, font="Cambria", bold=True,
         color="FFFFFF" if dark else INK)
    if sub:
        text(slide, 0.6, 1.1, 12.1, 0.45, [sub], size=15, color=MOSS if dark else MUTED)


def fit_image(slide, path, x, y, w, h):
    from PIL import Image
    iw, ih = Image.open(path).size
    sc = min(w / iw, h / ih)
    return slide.shapes.add_picture(str(path), Inches(x + (w - iw * sc) / 2), Inches(y + (h - ih * sc) / 2), width=Inches(iw * sc))


def card(slide, x, y, w, h, fill=LIGHT, line=None):
    sh = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(x), Inches(y), Inches(w), Inches(h))
    sh.adjustments[0] = 0.06
    sh.fill.solid()
    sh.fill.fore_color.rgb = rgb(fill)
    if line:
        sh.line.color.rgb = rgb(line)
        sh.line.width = Pt(1)
    else:
        sh.line.fill.background()
    sh.shadow.inherit = False
    return sh


def shape_text(sh, t, size, color, bold=True):
    tf = sh.text_frame
    tf.word_wrap = True
    tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    tf.margin_left = tf.margin_right = Inches(0.06)
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.CENTER
    r = p.add_run()
    r.text = t
    style_run(r, size=size, color=color, bold=bold)


def circle_tag(slide, x, y, d, t, fill=FOREST, size=14):
    sh = slide.shapes.add_shape(MSO_SHAPE.OVAL, Inches(x), Inches(y), Inches(d), Inches(d))
    sh.fill.solid(); sh.fill.fore_color.rgb = rgb(fill); sh.line.fill.background(); sh.shadow.inherit = False
    shape_text(sh, t, size, "FFFFFF")
    return sh


def stat(slide, x, y, w, number, label, color=FOREST, nsize=38, lsize=12.5, dark=False):
    text(slide, x, y, w, 0.72, [number], size=nsize, font="Cambria", bold=True, color=color)
    text(slide, x, y + 0.74, w, 0.7, [label], size=lsize, color="D6E2D0" if dark else MUTED)


def table(slide, rows, x, y, w, h, widths, size=13):
    tbl = slide.shapes.add_table(len(rows), len(rows[0]), Inches(x), Inches(y), Inches(w), Inches(h)).table
    for ci, wd_ in enumerate(widths):
        tbl.columns[ci].width = Inches(wd_)
    tbl.horz_banding = False
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            cell = tbl.cell(ri, ci)
            cell.fill.solid()
            cell.fill.fore_color.rgb = rgb(FOREST if ri == 0 else ("FFFFFF" if ri % 2 else LIGHT))
            tf = cell.text_frame
            p = tf.paragraphs[0]
            r = p.add_run()
            r.text = val
            style_run(r, size=size, color="FFFFFF" if ri == 0 else INK, bold=ri == 0 or ci == 0)
    return tbl


def chart_fonts(ch, size=12):
    ch.font.name = KO_FONT if CUR["lang"] == "ko" else "Calibri"
    ch.font.size = Pt(size)
    ch.font.color.rgb = rgb(INK)


def chart_title(ch, t, size=14):
    ch.has_title = True
    tf = ch.chart_title.text_frame
    tf.text = t
    r = tf.paragraphs[0].runs[0]
    style_run(r, size=size, bold=True)


def notes(slide, t):
    slide.notes_slide.notes_text_frame.text = t


def rich(s_txt, **o):
    """'A_ij', 'J_ang', 'NACH_r' 같은 토큰을 아래첨자 run으로 바꾼 run 목록."""
    import re
    runs, pos = [], 0
    for m_ in re.finditer(r"\b([A-Za-z]+)_([a-z]+)\b", s_txt):
        if m_.start() > pos:
            runs.append((s_txt[pos:m_.start()], dict(o)))
        runs.append((m_.group(1), dict(o)))
        runs.append((m_.group(2), dict(o, sub=True)))
        pos = m_.end()
    if pos < len(s_txt):
        runs.append((s_txt[pos:], dict(o)))
    return runs

## 12-9. 덱 만들기

In [11]:
def build_deck(lang):
    CUR["lang"] = lang
    s_ = S[lang]
    FL = FIG / lang
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
    blank = prs.slide_layouts[6]

    def new(dark=False):
        sl = prs.slides.add_slide(blank)
        sl.background.fill.solid()
        sl.background.fill.fore_color.rgb = rgb(DARK if dark else "FFFFFF")
        return sl

    NOTES = s_["notes"]
    # 1. title
    sl = new(dark=True)
    text(sl, 0.7, 1.3, 6.7, 2.2, s_["title"], size=40 if lang == "en" else 36, font="Cambria", bold=True, color="FFFFFF", space=0, line=1.05)
    text(sl, 0.7, 3.85, 6.3, 1.1, [s_["subtitle"]], size=17, color=MOSS)
    text(sl, 0.7, 5.5, 6.3, 0.9, s_["footer"], size=12, color="B8C7B5", space=4)
    fit_image(sl, FIG / "common" / "title_map.png", 7.5, 0.35, 5.5, 6.8)
    notes(sl, NOTES[0])
    # 2. close vs connected
    sl = new()
    title(sl, s_["s2_title"], sub=s_["s2_why"])
    text(sl, 0.6, 1.8, 5.3, 3.4, s_["s2_body"], size=16, bullets=True, space=11)
    card(sl, 0.6, 5.35, 5.3, 1.35)
    text(sl, 0.8, 5.5, 4.9, 1.1, [[(s_["s2_note"], {"italic": lang == "en"})]], size=13, color=MUTED)
    fit_image(sl, FIG / "common" / "close_vs_connected.png", 6.25, 1.6, 6.6, 3.5)
    text(sl, 6.25, 5.15, 3.0, 0.8, [s_["s2_caps"][0]], size=13, color=ISO, bold=True, align=PP_ALIGN.CENTER)
    text(sl, 9.6, 5.15, 3.25, 0.8, [s_["s2_caps"][1]], size=13, color=FOREST, bold=True, align=PP_ALIGN.CENTER)
    notes(sl, NOTES[1])
    # 3. definitions overview
    sl = new()
    title(sl, s_["s3_title"])
    for k, (tag, name, desc, analog) in enumerate(s_["s3_defs"]):
        x0 = 0.6 + k * 3.07
        card(sl, x0, 1.35, 2.88, 4.8, fill="F3F5F1" if tag == "D" else LIGHT)
        fit_image(sl, FIG / "common" / f"def_{tag}.png", x0 + 0.34, 1.45, 2.2, 2.1)
        circle_tag(sl, x0 + 0.18, 3.65, 0.5, tag, fill=MUTED if tag == "D" else FOREST, size=12)
        text(sl, x0 + 0.78, 3.7, 2.05, 0.5, [name], size=15.5, bold=True, color=MUTED if tag == "D" else FOREST)
        text(sl, x0 + 0.2, 4.28, 2.5, 1.2, [desc], size=12.5, color=INK, line=1.05)
        text(sl, x0 + 0.2, 5.48, 2.5, 0.6, [[(analog, {"italic": lang == "en", "color": ACCENT})]], size=11.5)
    card(sl, 0.6, 6.3, 12.1, 0.7, fill=FOREST)
    text(sl, 0.85, 6.3, 11.6, 0.7, [s_["s3_rule"]], size=15, color="FFFFFF", bold=True, anchor=MSO_ANCHOR.MIDDLE)
    notes(sl, NOTES[2])
    # 4. data
    sl = new()
    title(sl, s_["s4_title"])
    for k, (num, lab) in enumerate(s_["s4_stats"]):
        stat(sl, 0.6, 1.45 + k * 1.75, 5.2, num, lab, lsize=13.5)
    fit_image(sl, FIG / "common" / "network_parks.png", 6.6, 1.35, 6.2, 5.3)
    text(sl, 6.6, 6.7, 6.2, 0.4, [s_["s4_cap"]], size=12, color=MUTED, align=PP_ALIGN.CENTER)
    notes(sl, NOTES[3])
    # 5. counting turns
    sl = new()
    title(sl, s_["s5_title"])
    fit_image(sl, FIG / "common" / "turn_icons.png", 0.6, 1.4, 5.6, 1.9)
    text(sl, 0.6, 3.5, 5.6, 3.5, s_["s5_body"], size=15, bullets=True, space=9)
    fit_image(sl, FL / "angular_field.png", 6.5, 1.3, 6.4, 5.9)
    notes(sl, NOTES[4])
    # 6. attachment
    sl = new()
    title(sl, s_["s6_title"])
    fit_image(sl, FL / "attachment.png", 0.6, 1.35, 5.8, 5.8)
    text(sl, 6.9, 1.8, 5.8, 4.5, s_["s6_body"], size=17, bullets=True, space=16)
    notes(sl, NOTES[5])
    # D results (baseline)
    sl = new()
    title(sl, s_["sD_title"])
    fit_image(sl, FL / "D_overview.png", 0.4, 1.25, 4.9, 5.6)
    fit_image(sl, FL / "D_zoom.png", 5.3, 1.25, 5.3, 5.6)
    for k, (num, lab) in enumerate(s_["sD_stats"]):
        stat(sl, 10.8, 1.35 + k * 1.38, 2.35, num, lab, color=ISO if k == 3 else MUTED, nsize=34)
    text(sl, 0.6, 6.85, 10.0, 0.4, [s_["sD_cap"]], size=12, color=MUTED)
    notes(sl, s_["sD_note"])
    # 7. N1 results
    sl = new()
    title(sl, s_["s7_title"])
    fit_image(sl, FL / "N1_overview.png", 0.4, 1.25, 4.9, 5.6)
    fit_image(sl, FL / "N1_zoom.png", 5.3, 1.25, 5.3, 5.6)
    for k, (num, lab) in enumerate(s_["s7_stats"]):
        stat(sl, 10.8, 1.35 + k * 1.38, 2.35, num, lab, color=ISO if k == 3 else FOREST, nsize=34)
    text(sl, 0.6, 6.85, 10.0, 0.4, [s_["s7_cap"]], size=12, color=MUTED)
    notes(sl, NOTES[6])
    # 8. N2 results
    sl = new()
    title(sl, s_["s8_title"])
    fit_image(sl, FL / "N2_overview.png", 0.4, 1.25, 4.9, 5.9)
    fit_image(sl, FL / "N2_zoom.png", 5.3, 1.25, 5.3, 5.6)
    for k, (num, lab) in enumerate(s_["s8_stats"]):
        stat(sl, 10.8, 1.35 + k * 1.45, 2.35, num, lab, nsize=34)
    text(sl, 10.8, 4.4, 2.35, 2.8, s_["s8_body"], size=12.5, color=MUTED, space=10)
    notes(sl, NOTES[7])
    # 9. N3 results
    sl = new()
    title(sl, s_["s9_title"])
    fit_image(sl, FL / "N3_overview.png", 0.4, 1.25, 4.9, 5.9)
    fit_image(sl, FL / "N3_zoom.png", 5.3, 1.25, 5.3, 5.6)
    for k, (num, lab) in enumerate(s_["s9_stats"]):
        stat(sl, 10.8, 1.35 + k * 1.45, 2.35, num, lab, nsize=34)
    text(sl, 10.8, 4.4, 2.35, 2.8, s_["s9_body"], size=12.5, color=MUTED, space=10)
    notes(sl, NOTES[8])
    # 10. 10-minute rule
    sl = new()
    title(sl, s_["s10_title"])
    fit_image(sl, FL / "walk_split.png", 0.4, 1.3, 6.0, 5.9)
    text(sl, 6.8, 1.45, 5.9, 3.0, s_["s10_body"], size=15.5, bullets=True, space=10)
    rows = [s_["s10_head"]]
    for lab, a_, b_ in zip(s_["s10_rows"], [FDn, F2n, FN2n, FN3n], [FD, F2, FN2, FN3]):
        rows.append([lab, f"{int(a_.groups)} → {int(b_.groups)}", f"{int(a_.size_max)} → {int(b_.size_max)}"])
    table(sl, rows, 6.8, 4.55, 5.9, 2.1, [2.5, 1.7, 1.7], size=13.5)
    notes(sl, NOTES[9])
    # 11. distance?
    sl = new()
    title(sl, s_["s11_title"])
    cd = CategoryChartData()
    cd.categories = s_["s11_cats"]
    cd.add_series(s_["s11_series"], [float(ALSO_D[d]) for d in ["N1", "N2", "N3"]])
    ch = sl.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.6), Inches(1.4), Inches(6.6), Inches(5.4), cd).chart
    chart_fonts(ch, 13)
    chart_title(ch, s_["s11_series"], 14)
    ch.has_legend = False
    va = ch.value_axis
    va.minimum_scale, va.maximum_scale = 0, 1
    va.tick_labels.number_format = "0%"; va.tick_labels.number_format_is_linked = False
    va.major_gridlines.format.line.color.rgb = rgb("E3E6E1"); va.format.line.fill.background()
    pl = ch.plots[0]
    pl.gap_width = 80
    pl.series[0].format.fill.solid(); pl.series[0].format.fill.fore_color.rgb = rgb(FOREST)
    pl.has_data_labels = True
    pl.data_labels.number_format = "0%"; pl.data_labels.number_format_is_linked = False
    pl.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END; pl.data_labels.font.size = Pt(14)
    text(sl, 7.7, 1.6, 5.0, 5.2, s_["s11_body"], size=15.5, bullets=True, space=12)
    notes(sl, NOTES[10])
    # 12. isolated parks
    sl = new()
    title(sl, s_["s12_title"])
    fit_image(sl, FL / "cross_status.png", 0.4, 1.25, 5.0, 5.9)
    cd = CategoryChartData()
    cd.categories = [BORO[lang][b] for b in ISO_B.index]
    cd.add_series(s_["s12_chart"], [int(v) for v in ISO_B.values])
    ch = sl.shapes.add_chart(XL_CHART_TYPE.BAR_CLUSTERED, Inches(5.8), Inches(1.35), Inches(6.9), Inches(3.3), cd).chart
    chart_fonts(ch, 12)
    chart_title(ch, s_["s12_chart"], 14)
    ch.has_legend = False
    ch.value_axis.visible = False; ch.value_axis.has_major_gridlines = False
    ch.category_axis.reverse_order = True; ch.category_axis.format.line.color.rgb = rgb("BFC5BF")
    pl = ch.plots[0]
    pl.gap_width = 60
    pl.series[0].format.fill.solid(); pl.series[0].format.fill.fore_color.rgb = rgb(ISO)
    pl.has_data_labels = True; pl.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END; pl.data_labels.font.size = Pt(12)
    text(sl, 5.8, 4.9, 6.9, 2.2, s_["s12_body"], size=15, bullets=True, space=8)
    notes(sl, NOTES[11])
    # next step: commercial activity
    sl = new()
    title(sl, s_["s14_title"])
    fit_image(sl, FL / "rings.png", 0.4, 1.3, 6.0, 5.9)
    text(sl, 6.8, 1.5, 5.9, 3.6, s_["s14_body"], size=15, bullets=True, space=10)
    card(sl, 6.8, 5.3, 5.9, 1.4)
    text(sl, 7.0, 5.42, 5.5, 1.2, [s_["s14_card"]], size=13.5, color=FOREST, bold=True, anchor=MSO_ANCHOR.MIDDLE)
    notes(sl, s_["s14_note"])
    # 13. takeaways
    sl = new(dark=True)
    title(sl, s_["s13_title"], dark=True)
    for k, (h_, d_) in enumerate(s_["s13_cards"]):
        x0 = 0.6 + k * 4.12
        card(sl, x0, 1.6, 3.85, 3.0, fill="24392E")
        text(sl, x0 + 0.28, 1.85, 3.3, 2.6, [[(h_, {"bold": True, "size": 20, "color": "FFFFFF"})], [(d_, {"size": 15, "color": "D6E2D0"})]], space=10)
    card(sl, 0.6, 5.1, 12.1, 0.9, fill="24392E")
    text(sl, 0.9, 5.1, 11.5, 0.9, [s_["s13_next"]], size=17, color=MOSS, bold=True, anchor=MSO_ANCHOR.MIDDLE)
    notes(sl, NOTES[12])
    # ===== appendix =====
    sl = new(dark=True)
    title(sl, s_["app_title"], dark=True)
    text(sl, 0.6, 1.6, 12.0, 5.0, s_["app_items"], size=18, color="D6E2D0", bullets=True, space=10)
    notes(sl, NOTES[13])
    # A1 pipeline
    sl = new()
    title(sl, s_["a1_title"])
    bw, gap, y0 = 1.78, 0.29, 1.55
    for k, st in enumerate(s_["a1_steps"]):
        x0 = 0.6 + k * (bw + gap)
        sh = card(sl, x0, y0, bw, 1.2, fill=FOREST if k in (3, 4) else "FFFFFF", line=None if k in (3, 4) else FOREST)
        shape_text(sh, st, 13, "FFFFFF" if k in (3, 4) else FOREST)
        if k < len(s_["a1_steps"]) - 1:
            ar = sl.shapes.add_shape(MSO_SHAPE.RIGHT_ARROW, Inches(x0 + bw + 0.04), Inches(y0 + 0.47), Inches(0.21), Inches(0.25))
            ar.fill.solid(); ar.fill.fore_color.rgb = rgb(MOSS); ar.line.fill.background(); ar.shadow.inherit = False
    text(sl, 0.6, 3.2, 12.1, 3.8, s_["a1_body"], size=14.5, bullets=True, space=12)
    # A2 network & attachment
    sl = new()
    title(sl, s_["a2_title"])
    fit_image(sl, FL / "network_S_vs_C.png", 0.4, 1.3, 7.6, 4.2)
    text(sl, 8.3, 1.4, 4.5, 5.6, s_["a2_body"], size=13, bullets=True, space=10)
    # A3 angular cost
    sl = new()
    title(sl, s_["a3_title"])
    fit_image(sl, FL / "concept_angular.png", 0.6, 1.4, 12.1, 3.4)
    card(sl, 0.6, 5.05, 12.1, 1.75)
    text(sl, 0.85, 5.2, 11.6, 1.5, [[(h_, {"bold": True, "color": FOREST}), (d_, {})] for h_, d_ in s_["a3_body"]], size=13.5, space=8)
    # A4a D formal
    sl = new()
    title(sl, s_["ad_title"])
    fit_image(sl, FIG / "common" / "def_D.png", 0.8, 1.5, 4.6, 4.6)
    card(sl, 6.2, 1.5, 6.5, 1.35)
    text(sl, 6.45, 1.62, 6.0, 1.15, [rich(s_["ad_eq"], bold=True), [(s_["ad_eq_note"], {"size": 12, "color": MUTED})]], size=16, space=4)
    text(sl, 6.2, 3.15, 6.5, 3.8, s_["ad_body"], size=14, bullets=True, space=10)
    # A4 N1 formal
    sl = new()
    title(sl, s_["a4_title"])
    fit_image(sl, FL / "concept_N1.png", 0.5, 1.4, 5.4, 5.6)
    card(sl, 6.2, 1.45, 6.5, 1.25)
    eq1, eq2 = s_["a4_eq"]
    text(sl, 6.45, 1.58, 6.0, 1.05, [rich(eq1), rich(eq2, bold=True)], size=14.5, space=4)
    table(sl, s_["a4_table"], 6.2, 2.95, 6.5, 2.0, [1.0, 3.2, 2.3], size=13)
    text(sl, 6.2, 5.2, 6.5, 2.0, s_["a4_body"], size=12, color=MUTED, bullets=True, space=4)
    # A5 N2 formal
    sl = new()
    title(sl, s_["a5_title"])
    fit_image(sl, FL / "concept_N2.png", 0.4, 1.4, 6.6, 5.4)
    card(sl, 7.2, 1.45, 5.5, 1.3)
    text(sl, 7.45, 1.58, 5.1, 1.1, [rich(s_["a5_eq"], bold=True), [(s_["a5_eq_note"], {"size": 12, "color": MUTED})]], size=16, space=4)
    text(sl, 7.2, 3.0, 5.5, 4.2, [rich(b_) for b_ in s_["a5_body"]], size=13, bullets=True, space=7)
    # A6 N3 formal
    sl = new()
    title(sl, s_["a6_title"])
    fit_image(sl, FL / "concept_N3.png", 0.4, 1.35, 6.4, 5.7)
    text(sl, 7.2, 1.5, 5.5, 3.6, [rich(b_) for b_ in s_["a6_body"]], size=14, bullets=True, space=8)
    card(sl, 7.2, 5.15, 5.5, 1.5)
    text(sl, 7.4, 5.28, 5.1, 1.3, [s_["a6_cap"]], size=12.5)
    # A7 percolation
    sl = new()
    title(sl, s_["a7_title"])
    cd = CategoryChartData()
    cd.categories = [f"{t}°" for t in PERC.index]
    for R in PERC.columns:
        cd.add_series(s_["a7_series"].format(R=R), [float(v) for v in PERC[R].values])
    ch = sl.shapes.add_chart(XL_CHART_TYPE.LINE_MARKERS, Inches(0.5), Inches(1.35), Inches(6.6), Inches(5.4), cd).chart
    chart_fonts(ch, 12)
    chart_title(ch, s_["a7_chart"], 13)
    ch.has_legend = True; ch.legend.position = XL_LEGEND_POSITION.BOTTOM; ch.legend.include_in_layout = False
    va = ch.value_axis
    va.minimum_scale, va.maximum_scale = 0, 1
    va.tick_labels.number_format = "0%"; va.tick_labels.number_format_is_linked = False
    va.major_gridlines.format.line.color.rgb = rgb("E3E6E1"); va.format.line.fill.background()
    for ser, c in zip(ch.plots[0].series, [MOSS, FOREST, DARK]):
        ser.format.line.color.rgb = rgb(c); ser.format.line.width = Pt(2.75); ser.smooth = False
        ser.marker.style = XL_MARKER_STYLE.CIRCLE; ser.marker.size = 8
        ser.marker.format.fill.solid(); ser.marker.format.fill.fore_color.rgb = rgb(c); ser.marker.format.line.color.rgb = rgb(c)
    text(sl, 7.5, 1.5, 5.3, 5.3, [[(h_, {"bold": True, "color": ACCENT if k == 0 else (ISO if k == 3 else FOREST)}), (d_, {})]
                                   for k, (h_, d_) in enumerate(s_["a7_body"])], size=13.5, space=12)
    # A8 AUC
    sl = new()
    title(sl, s_["a8_title"])
    cd = CategoryChartData()
    cd.categories = s_["a8_cats"]
    cd.add_series(s_["a8_series"][0], [float(v) for v in AUC.median_AUC.values])
    cd.add_series(s_["a8_series"][1], [float(v) for v in AUC.median_AUC_all.values])
    ch = sl.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.5), Inches(1.4), Inches(7.4), Inches(5.3), cd).chart
    chart_fonts(ch, 12)
    ch.has_legend = True; ch.legend.position = XL_LEGEND_POSITION.BOTTOM; ch.legend.include_in_layout = False
    va = ch.value_axis
    va.minimum_scale, va.maximum_scale = 0.5, 1.0
    va.major_gridlines.format.line.color.rgb = rgb("E3E6E1"); va.format.line.fill.background()
    va.tick_labels.number_format = "0.00"; va.tick_labels.number_format_is_linked = False
    pl = ch.plots[0]
    pl.gap_width = 70; pl.overlap = -5
    for ser, c in zip(pl.series, [FOREST, "C9D6C2"]):
        ser.format.fill.solid(); ser.format.fill.fore_color.rgb = rgb(c)
    pl.has_data_labels = True
    pl.data_labels.number_format = "0.00"; pl.data_labels.number_format_is_linked = False
    pl.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END; pl.data_labels.font.size = Pt(11)
    card(sl, 8.3, 1.5, 4.4, 3.6)
    text(sl, 8.55, 1.7, 3.95, 3.3, [[(h_, {"bold": True}), (d_, {})] for h_, d_ in s_["a8_body"]], size=13.5, space=10)
    # A9 limitations
    sl = new()
    title(sl, s_["a9_title"])
    text(sl, 0.6, 1.5, 12.1, 5.5, s_["a9_body"], size=16, bullets=True, space=12)
    out = SL_DIR / f"NYC_park_networks_{lang.upper()}.pptx"
    prs.save(out)
    CUR["lang"] = "en"
    return out, len(prs.slides)


for lang in LANGS:
    print(build_deck(lang))

(PosixPath('/Users/sunghosynn/Documents/Code/Park_Network/outputs/NYC/slides/NYC_park_networks_EN.pptx'), 26)


(PosixPath('/Users/sunghosynn/Documents/Code/Park_Network/outputs/NYC/slides/NYC_park_networks_KO.pptx'), 26)
